In [1]:
#!/usr/bin/env python3
"""
PPS Experiment – XDF Stream Extractor (Dependency-Free Version)

This script loads a LabRecorder .xdf file, finds the Audio and MouseEvents streams,
and saves them as WAV and CSV files without relying on pandas, soundfile, or matplotlib.

Usage:
    python xdf_extractor.py
    
Or modify the paths in the USER CONFIG section below.
"""

import os
import pathlib
import re
import json
import struct
import wave
import csv
from typing import Optional, List, Dict, Any
import xml.etree.ElementTree as ET

# Try to import numpy, with fallback
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    print("Warning: NumPy not available. Using pure Python fallbacks.")
    HAS_NUMPY = False

# ============================================================================
# SIMPLE XDF LOADER (bypasses pyxdf dependency)
# ============================================================================

def read_xdf_simple(filepath: str) -> tuple:
    """
    Simplified XDF reader that extracts basic stream information.
    Returns (streams, header) where streams is a list of stream dictionaries.
    """
    streams = []
    
    with open(filepath, 'rb') as f:
        # Read XDF header
        header = {}
        
        while True:
            # Read chunk header
            try:
                length_bytes = f.read(4)
                if len(length_bytes) < 4:
                    break
                length = struct.unpack('<I', length_bytes)[0]
                
                if length == 0:
                    continue
                    
                tag = struct.unpack('<H', f.read(2))[0]
                payload = f.read(length - 2)
                
                if tag == 2:  # Stream header
                    stream_info = parse_stream_header(payload)
                    stream = {
                        'info': stream_info,
                        'time_series': [],
                        'time_stamps': []
                    }
                    streams.append(stream)
                    
                elif tag == 3:  # Samples
                    if streams:
                        parse_samples(payload, streams[-1])
                        
                elif tag == 6:  # Stream footer
                    continue
                    
            except struct.error:
                break
                
    return streams, header

def parse_stream_header(payload: bytes) -> dict:
    """Parse stream header XML."""
    try:
        xml_str = payload.decode('utf-8')
        root = ET.fromstring(xml_str)
        
        info = {}
        for elem in root:
            if len(list(elem)) == 0:  # Leaf node
                info[elem.tag] = [elem.text or '']
            else:  # Has children
                info[elem.tag] = {}
                for child in elem:
                    info[elem.tag][child.tag] = [child.text or '']
                    
        return info
    except:
        return {'name': ['Unknown'], 'type': ['Unknown'], 'channel_count': ['1']}

def parse_samples(payload: bytes, stream: dict) -> None:
    """Parse sample data from payload."""
    try:
        channel_count = int(stream['info'].get('channel_count', ['1'])[0])
        sample_format = stream['info'].get('channel_format', ['float32'])[0]
        
        # Simple parsing - assumes float32 format
        if sample_format == 'float32':
            sample_size = 4
        else:
            sample_size = 4  # Default
            
        # Each sample has timestamp (8 bytes) + channel data
        timestamp_size = 8
        record_size = timestamp_size + (channel_count * sample_size)
        
        offset = 0
        while offset + record_size <= len(payload):
            # Read timestamp
            timestamp = struct.unpack('<d', payload[offset:offset+8])[0]
            offset += 8
            
            # Read channel data
            sample_data = []
            for _ in range(channel_count):
                if sample_format == 'float32':
                    value = struct.unpack('<f', payload[offset:offset+4])[0]
                else:
                    value = 0.0
                sample_data.append(value)
                offset += 4
                
            stream['time_stamps'].append(timestamp)
            stream['time_series'].append(sample_data)
            
    except:
        pass  # Skip malformed samples

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def find_stream(streams: List[Dict], *, type_=None, name_regex=None, channel_count=None) -> Optional[Dict]:
    """Return the first stream that matches all given criteria."""
    for s in streams:
        info = s['info']
        name = info.get('name', [''])[0]
        stype = info.get('type', [''])[0]
        nch = int(info.get('channel_count', ['1'])[0])
        
        if type_ is not None and stype != type_:
            continue
        if channel_count is not None and nch != channel_count:
            continue
        if name_regex is not None and re.fullmatch(name_regex, name) is None:
            continue
        return s
    return None

def save_combined_csv(filename: str, audio_data: List[List[float]], audio_timestamps: List[float], 
                     sample_rate: int, mouse_data: List[List[float]], mouse_timestamps: List[float]) -> None:
    """Save combined audio and mouse data in a single synchronized CSV file."""
    
    # Process audio data
    if HAS_NUMPY:
        audio_samples = np.array(audio_data, dtype=np.float32)
        if audio_samples.ndim == 1:
            audio_samples = audio_samples.reshape(-1, 1)
        n_audio_frames = audio_samples.shape[0]
        n_channels = audio_samples.shape[1]
    else:
        audio_samples = audio_data
        if len(audio_data) > 0 and isinstance(audio_data[0], (int, float)):
            audio_samples = [[x] for x in audio_data]
        n_audio_frames = len(audio_samples)
        n_channels = len(audio_samples[0]) if len(audio_samples) > 0 else 1
    
    # Create synchronized time axis from audio (this is our master timeline)
    if audio_timestamps and len(audio_timestamps) == n_audio_frames:
        t0 = audio_timestamps[0]  # Reference time
        audio_time_axis = [t - t0 for t in audio_timestamps]
    else:
        audio_time_axis = [i / sample_rate for i in range(n_audio_frames)]
        t0 = 0
    
    # Process mouse clicks - convert to relative time and create lookup
    mouse_events = {}  # time -> (x, y)
    if mouse_data and mouse_timestamps:
        for i, (mouse_time, mouse_pos) in enumerate(zip(mouse_timestamps, mouse_data)):
            rel_time = mouse_time - t0
            # Round to nearest audio sample time for alignment
            closest_audio_idx = min(range(len(audio_time_axis)), 
                                  key=lambda j: abs(audio_time_axis[j] - rel_time))
            aligned_time = audio_time_axis[closest_audio_idx]
            mouse_events[aligned_time] = (mouse_pos[1], mouse_pos[2])  # x, y coordinates
    
    # Create CSV with combined data
    with open(filename, 'w', newline='') as csvfile:
        # Prepare headers based on audio channels
        if n_channels == 1:
            headers = ['time_sec', 'audio_amplitude', 'mouse_click', 'mouse_x', 'mouse_y']
        elif n_channels == 2:
            headers = ['time_sec', 'audio_left', 'audio_right', 'mouse_click', 'mouse_x', 'mouse_y']
        else:
            headers = ['time_sec'] + [f'audio_ch{j+1}' for j in range(n_channels)] + ['mouse_click', 'mouse_x', 'mouse_y']
        
        writer = csv.writer(csvfile)
        writer.writerow(headers)
        
        # Write synchronized data
        for i in range(n_audio_frames):
            time_point = audio_time_axis[i]
            row = [time_point]
            
            # Add audio data
            for j in range(n_channels):
                if HAS_NUMPY:
                    amplitude = float(audio_samples[i, j])
                else:
                    amplitude = audio_samples[i][j] if i < len(audio_samples) else 0.0
                row.append(amplitude)
            
            # Add mouse data (if click occurred at this time)
            if time_point in mouse_events:
                row.extend([1, mouse_events[time_point][0], mouse_events[time_point][1]])  # click=1, x, y
            else:
                row.extend([0, '', ''])  # no click, empty coordinates
            
            writer.writerow(row)

def save_csv_simple(filename: str, data: List[List[float]], headers: List[str]) -> None:
    """Save data as CSV file without pandas dependency."""
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)
        for row in data:
            writer.writerow(row)

# ============================================================================
# MAIN EXTRACTION LOGIC
# ============================================================================

def extract_xdf_streams(xdf_path: str, out_dir: str) -> None:
    """Main function to extract streams from XDF file."""
    out_dir = pathlib.Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Loading {xdf_path} ...")
    
    # Try pyxdf first, fall back to simple loader
    streams = None
    try:
        import pyxdf
        streams, _ = pyxdf.load_xdf(xdf_path)
        print(f"Loaded {len(streams)} streams using pyxdf")
    except ImportError:
        print("pyxdf not available, using simple XDF loader...")
        streams, _ = read_xdf_simple(xdf_path)
        print(f"Loaded {len(streams)} streams using simple loader")
    except Exception as e:
        print(f"pyxdf failed: {e}")
        print("Falling back to simple XDF loader...")
        streams, _ = read_xdf_simple(xdf_path)
        print(f"Loaded {len(streams)} streams using simple loader")
    
    if not streams:
        print("No streams found in XDF file!")
        return
    
    # Identify audio stream (2 channels, type 'Audio')
    audio_stream = (
        find_stream(streams, type_="Audio", channel_count=2)
        or find_stream(streams, name_regex=r".*_Audio_.*")
    )
    
    # Identify mouse-click stream (3 channels, type 'MouseEvents')
    mouse_stream = (
        find_stream(streams, type_="MouseEvents", channel_count=3)
        or find_stream(streams, name_regex=r".*_MouseClicks_.*")
    )
    
    print("\n--- streams detected ---")
    print("Audio:", audio_stream['info'].get('name', ['not found'])[0] if audio_stream else "not found")
    print("Mouse:", mouse_stream['info'].get('name', ['not found'])[0] if mouse_stream else "not found")
    
    # Save combined audio and mouse data
    if audio_stream:
        audio_name = audio_stream["info"]["name"][0]
        sr_str = audio_stream["info"].get("nominal_srate", ["44100"])[0]
        sr = int(float(sr_str))
        
        audio_ts = audio_stream["time_series"]
        audio_timestamps = audio_stream["time_stamps"]
        
        if len(audio_ts) == 0:
            print(f"⚠️  Audio stream '{audio_name}' contains 0 samples – nothing to write.")
            return
        
        # Get mouse data if available
        mouse_data = mouse_stream["time_series"] if mouse_stream else []
        mouse_timestamps = mouse_stream["time_stamps"] if mouse_stream else []
        mouse_name = mouse_stream['info']['name'][0] if mouse_stream else "NoMouse"
        
        # Create combined filename
        combined_name = f"{audio_name}_combined_data.csv"
        csv_path = out_dir / combined_name
        
        save_combined_csv(str(csv_path), audio_ts, audio_timestamps, sr, mouse_data, mouse_timestamps)
        
        print(f"✓ Saved combined data -> {csv_path}")
        print(f"  - Audio samples: {len(audio_ts)}, sr={sr}")
        print(f"  - Mouse clicks: {len(mouse_data) if mouse_data else 0}")
        
    else:
        print("No audio stream found - cannot create combined file")
        
        # Save mouse data separately if no audio
        if mouse_stream:
            name = mouse_stream['info']['name'][0]
            click_data = mouse_stream['time_series']
            
            if click_data:
                csv_path = out_dir / f"{name}_mouseonly.csv"
                headers = ['rel_time_sec', 'x_px', 'y_px']
                save_csv_simple(str(csv_path), click_data, headers)
                print(f"✓ Saved mouse-only data -> {csv_path}  rows={len(click_data)}")
            else:
                print("Mouse stream contains no data")

# ============================================================================
# SIMPLE VISUALIZATION (optional, without matplotlib)
# ============================================================================

def create_simple_plot(xdf_path: str) -> None:
    """Create a simple text-based visualization of the data."""
    print("\n--- Simple Data Analysis ---")
    
    try:
        import pyxdf
        streams, _ = pyxdf.load_xdf(xdf_path)
    except:
        streams, _ = read_xdf_simple(xdf_path)
    
    audio = find_stream(streams, type_="Audio")
    mouse = find_stream(streams, type_="MouseEvents")
    
    if audio and len(audio["time_series"]) > 0:
        if HAS_NUMPY:
            aud_data = np.array(audio["time_series"])
            aud_ts = np.array(audio["time_stamps"])
            duration = aud_ts[-1] - aud_ts[0] if len(aud_ts) > 1 else 0
        else:
            duration = (audio["time_stamps"][-1] - audio["time_stamps"][0] 
                       if len(audio["time_stamps"]) > 1 else 0)
        
        print(f"Audio: {len(audio['time_series'])} samples, {duration:.2f}s duration")
    
    if mouse and len(mouse["time_series"]) > 0:
        if HAS_NUMPY:
            click_ts = np.array(mouse["time_stamps"])
            t0 = audio["time_stamps"][0] if audio and audio["time_stamps"] else 0
            rel_clicks = click_ts - t0
            print(f"Mouse: {len(mouse['time_series'])} clicks")
            print(f"Click times (rel): {rel_clicks[:5]}..." if len(rel_clicks) > 5 else f"Click times (rel): {rel_clicks}")
        else:
            print(f"Mouse: {len(mouse['time_series'])} clicks")

# ============================================================================
# USER CONFIGURATION & MAIN
# ============================================================================

if __name__ == "__main__":
    # ---- USER INPUTS -------------------------------------------------------
    xdf_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf"
    out_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT"
    # -----------------------------------------------------------------------
    
    # Check if file exists
    if not os.path.exists(xdf_path):
        print(f"Error: XDF file not found: {xdf_path}")
        print("Please update the xdf_path variable in the script.")
        exit(1)
    
    # Extract streams
    extract_xdf_streams(xdf_path, out_dir)
    
    # Optional: create simple analysis
    create_simple_plot(xdf_path)
    
    print("\n✓ Processing complete!")

Loading C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf ...
Loaded 2 streams using pyxdf

--- streams detected ---
Audio: Audio_P001_20250506_171059
Mouse: MouseClicks_P001_20250506_171059


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [2]:
#!/usr/bin/env python3
"""
PPS Experiment – XDF Stream Extractor (Dependency-Free Version)

This script loads a LabRecorder .xdf file, finds the Audio and MouseEvents streams,
and saves them as WAV and CSV files without relying on pandas, soundfile, or matplotlib.

Usage:
    python xdf_extractor.py
    
Or modify the paths in the USER CONFIG section below.
"""

import os
import pathlib
import re
import json
import struct
import wave
import csv
from typing import Optional, List, Dict, Any
import xml.etree.ElementTree as ET

# Try to import numpy, with fallback
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    print("Warning: NumPy not available. Using pure Python fallbacks.")
    HAS_NUMPY = False

# ============================================================================
# SIMPLE XDF LOADER (bypasses pyxdf dependency)
# ============================================================================

def read_xdf_simple(filepath: str) -> tuple:
    """
    Simplified XDF reader that extracts basic stream information.
    Returns (streams, header) where streams is a list of stream dictionaries.
    """
    streams = []
    
    with open(filepath, 'rb') as f:
        # Read XDF header
        header = {}
        
        while True:
            # Read chunk header
            try:
                length_bytes = f.read(4)
                if len(length_bytes) < 4:
                    break
                length = struct.unpack('<I', length_bytes)[0]
                
                if length == 0:
                    continue
                    
                tag = struct.unpack('<H', f.read(2))[0]
                payload = f.read(length - 2)
                
                if tag == 2:  # Stream header
                    stream_info = parse_stream_header(payload)
                    stream = {
                        'info': stream_info,
                        'time_series': [],
                        'time_stamps': []
                    }
                    streams.append(stream)
                    
                elif tag == 3:  # Samples
                    if streams:
                        parse_samples(payload, streams[-1])
                        
                elif tag == 6:  # Stream footer
                    continue
                    
            except struct.error:
                break
                
    return streams, header

def parse_stream_header(payload: bytes) -> dict:
    """Parse stream header XML."""
    try:
        xml_str = payload.decode('utf-8')
        root = ET.fromstring(xml_str)
        
        info = {}
        for elem in root:
            if len(list(elem)) == 0:  # Leaf node
                info[elem.tag] = [elem.text or '']
            else:  # Has children
                info[elem.tag] = {}
                for child in elem:
                    info[elem.tag][child.tag] = [child.text or '']
                    
        return info
    except:
        return {'name': ['Unknown'], 'type': ['Unknown'], 'channel_count': ['1']}

def parse_samples(payload: bytes, stream: dict) -> None:
    """Parse sample data from payload."""
    try:
        channel_count = int(stream['info'].get('channel_count', ['1'])[0])
        sample_format = stream['info'].get('channel_format', ['float32'])[0]
        
        # Simple parsing - assumes float32 format
        if sample_format == 'float32':
            sample_size = 4
        else:
            sample_size = 4  # Default
            
        # Each sample has timestamp (8 bytes) + channel data
        timestamp_size = 8
        record_size = timestamp_size + (channel_count * sample_size)
        
        offset = 0
        while offset + record_size <= len(payload):
            # Read timestamp
            timestamp = struct.unpack('<d', payload[offset:offset+8])[0]
            offset += 8
            
            # Read channel data
            sample_data = []
            for _ in range(channel_count):
                if sample_format == 'float32':
                    value = struct.unpack('<f', payload[offset:offset+4])[0]
                else:
                    value = 0.0
                sample_data.append(value)
                offset += 4
                
            stream['time_stamps'].append(timestamp)
            stream['time_series'].append(sample_data)
            
    except:
        pass  # Skip malformed samples

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def find_stream(streams: List[Dict], *, type_=None, name_regex=None, channel_count=None) -> Optional[Dict]:
    """Return the first stream that matches all given criteria."""
    for s in streams:
        info = s['info']
        name = info.get('name', [''])[0]
        stype = info.get('type', [''])[0]
        nch = int(info.get('channel_count', ['1'])[0])
        
        if type_ is not None and stype != type_:
            continue
        if channel_count is not None and nch != channel_count:
            continue
        if name_regex is not None and re.fullmatch(name_regex, name) is None:
            continue
        return s
    return None

def save_combined_csv(filename: str, audio_data: List[List[float]], audio_timestamps: List[float], 
                     sample_rate: int, mouse_data: List[List[float]], mouse_timestamps: List[float]) -> None:
    """Save combined audio and mouse data in a single synchronized CSV file."""
    
    # Process audio data
    if HAS_NUMPY:
        audio_samples = np.array(audio_data, dtype=np.float32)
        if audio_samples.ndim == 1:
            audio_samples = audio_samples.reshape(-1, 1)
        n_audio_frames = audio_samples.shape[0]
        n_channels = audio_samples.shape[1]
    else:
        audio_samples = audio_data
        if len(audio_data) > 0 and isinstance(audio_data[0], (int, float)):
            audio_samples = [[x] for x in audio_data]
        n_audio_frames = len(audio_samples)
        n_channels = len(audio_samples[0]) if len(audio_samples) > 0 else 1
    
    # Create synchronized time axis from audio (this is our master timeline)
    if audio_timestamps is not None and len(audio_timestamps) == n_audio_frames:
        t0 = audio_timestamps[0]  # Reference time
        audio_time_axis = [t - t0 for t in audio_timestamps]
    else:
        audio_time_axis = [i / sample_rate for i in range(n_audio_frames)]
        t0 = 0
    
    # Process mouse clicks - convert to relative time and create lookup
    mouse_events = {}  # time -> (x, y)
    if mouse_data is not None and mouse_timestamps is not None and len(mouse_data) > 0:
        for i, (mouse_time, mouse_pos) in enumerate(zip(mouse_timestamps, mouse_data)):
            rel_time = mouse_time - t0
            # Round to nearest audio sample time for alignment
            closest_audio_idx = min(range(len(audio_time_axis)), 
                                  key=lambda j: abs(audio_time_axis[j] - rel_time))
            aligned_time = audio_time_axis[closest_audio_idx]
            mouse_events[aligned_time] = (mouse_pos[1], mouse_pos[2])  # x, y coordinates
    
    # Create CSV with combined data
    with open(filename, 'w', newline='') as csvfile:
        # Prepare headers based on audio channels
        if n_channels == 1:
            headers = ['time_sec', 'audio_amplitude', 'mouse_click', 'mouse_x', 'mouse_y']
        elif n_channels == 2:
            headers = ['time_sec', 'audio_left', 'audio_right', 'mouse_click', 'mouse_x', 'mouse_y']
        else:
            headers = ['time_sec'] + [f'audio_ch{j+1}' for j in range(n_channels)] + ['mouse_click', 'mouse_x', 'mouse_y']
        
        writer = csv.writer(csvfile)
        writer.writerow(headers)
        
        # Write synchronized data
        for i in range(n_audio_frames):
            time_point = audio_time_axis[i]
            row = [time_point]
            
            # Add audio data
            for j in range(n_channels):
                if HAS_NUMPY:
                    amplitude = float(audio_samples[i, j])
                else:
                    amplitude = audio_samples[i][j] if i < len(audio_samples) else 0.0
                row.append(amplitude)
            
            # Add mouse data (if click occurred at this time)
            if time_point in mouse_events:
                row.extend([1, mouse_events[time_point][0], mouse_events[time_point][1]])  # click=1, x, y
            else:
                row.extend([0, '', ''])  # no click, empty coordinates
            
            writer.writerow(row)

def save_csv_simple(filename: str, data: List[List[float]], headers: List[str]) -> None:
    """Save data as CSV file without pandas dependency."""
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)
        for row in data:
            writer.writerow(row)

# ============================================================================
# MAIN EXTRACTION LOGIC
# ============================================================================

def extract_xdf_streams(xdf_path: str, out_dir: str) -> None:
    """Main function to extract streams from XDF file."""
    out_dir = pathlib.Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Loading {xdf_path} ...")
    
    # Try pyxdf first, fall back to simple loader
    streams = None
    try:
        import pyxdf
        streams, _ = pyxdf.load_xdf(xdf_path)
        print(f"Loaded {len(streams)} streams using pyxdf")
    except ImportError:
        print("pyxdf not available, using simple XDF loader...")
        streams, _ = read_xdf_simple(xdf_path)
        print(f"Loaded {len(streams)} streams using simple loader")
    except Exception as e:
        print(f"pyxdf failed: {e}")
        print("Falling back to simple XDF loader...")
        streams, _ = read_xdf_simple(xdf_path)
        print(f"Loaded {len(streams)} streams using simple loader")
    
    if not streams:
        print("No streams found in XDF file!")
        return
    
    # Identify audio stream (2 channels, type 'Audio')
    audio_stream = (
        find_stream(streams, type_="Audio", channel_count=2)
        or find_stream(streams, name_regex=r".*_Audio_.*")
    )
    
    # Identify mouse-click stream (3 channels, type 'MouseEvents')
    mouse_stream = (
        find_stream(streams, type_="MouseEvents", channel_count=3)
        or find_stream(streams, name_regex=r".*_MouseClicks_.*")
    )
    
    print("\n--- streams detected ---")
    print("Audio:", audio_stream['info'].get('name', ['not found'])[0] if audio_stream else "not found")
    print("Mouse:", mouse_stream['info'].get('name', ['not found'])[0] if mouse_stream else "not found")
    
    # Save combined audio and mouse data
    if audio_stream:
        audio_name = audio_stream["info"]["name"][0]
        sr_str = audio_stream["info"].get("nominal_srate", ["44100"])[0]
        sr = int(float(sr_str))
        
        audio_ts = audio_stream["time_series"]
        audio_timestamps = audio_stream["time_stamps"]
        
        if len(audio_ts) == 0:
            print(f"⚠️  Audio stream '{audio_name}' contains 0 samples – nothing to write.")
            return
        
        # Get mouse data if available
        mouse_data = mouse_stream["time_series"] if mouse_stream is not None else []
        mouse_timestamps = mouse_stream["time_stamps"] if mouse_stream is not None else []
        mouse_name = mouse_stream['info']['name'][0] if mouse_stream is not None else "NoMouse"
        
        # Create combined filename
        combined_name = f"{audio_name}_combined_data.csv"
        csv_path = out_dir / combined_name
        
        save_combined_csv(str(csv_path), audio_ts, audio_timestamps, sr, mouse_data, mouse_timestamps)
        
        print(f"✓ Saved combined data -> {csv_path}")
        print(f"  - Audio samples: {len(audio_ts)}, sr={sr}")
        print(f"  - Mouse clicks: {len(mouse_data) if mouse_data else 0}")
        
    else:
        print("No audio stream found - cannot create combined file")
        
        # Save mouse data separately if no audio
        if mouse_stream is not None:
            name = mouse_stream['info']['name'][0]
            click_data = mouse_stream['time_series']
            
            if click_data is not None and len(click_data) > 0:
                csv_path = out_dir / f"{name}_mouseonly.csv"
                headers = ['rel_time_sec', 'x_px', 'y_px']
                save_csv_simple(str(csv_path), click_data, headers)
                print(f"✓ Saved mouse-only data -> {csv_path}  rows={len(click_data)}")
            else:
                print("Mouse stream contains no data")

# ============================================================================
# SIMPLE VISUALIZATION (optional, without matplotlib)
# ============================================================================

def create_simple_plot(xdf_path: str) -> None:
    """Create a simple text-based visualization of the data."""
    print("\n--- Simple Data Analysis ---")
    
    try:
        import pyxdf
        streams, _ = pyxdf.load_xdf(xdf_path)
    except:
        streams, _ = read_xdf_simple(xdf_path)
    
    audio = find_stream(streams, type_="Audio")
    mouse = find_stream(streams, type_="MouseEvents")
    
    if audio and len(audio["time_series"]) > 0:
        if HAS_NUMPY:
            aud_data = np.array(audio["time_series"])
            aud_ts = np.array(audio["time_stamps"])
            duration = aud_ts[-1] - aud_ts[0] if len(aud_ts) > 1 else 0
        else:
            duration = (audio["time_stamps"][-1] - audio["time_stamps"][0] 
                       if len(audio["time_stamps"]) > 1 else 0)
        
        print(f"Audio: {len(audio['time_series'])} samples, {duration:.2f}s duration")
    
    if mouse and len(mouse["time_series"]) > 0:
        if HAS_NUMPY:
            click_ts = np.array(mouse["time_stamps"])
            t0 = audio["time_stamps"][0] if audio and audio["time_stamps"] else 0
            rel_clicks = click_ts - t0
            print(f"Mouse: {len(mouse['time_series'])} clicks")
            print(f"Click times (rel): {rel_clicks[:5]}..." if len(rel_clicks) > 5 else f"Click times (rel): {rel_clicks}")
        else:
            print(f"Mouse: {len(mouse['time_series'])} clicks")

# ============================================================================
# USER CONFIGURATION & MAIN
# ============================================================================

if __name__ == "__main__":
    # ---- USER INPUTS -------------------------------------------------------
    xdf_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf"
    out_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT"
    # -----------------------------------------------------------------------
    
    # Check if file exists
    if not os.path.exists(xdf_path):
        print(f"Error: XDF file not found: {xdf_path}")
        print("Please update the xdf_path variable in the script.")
        exit(1)
    
    # Extract streams
    extract_xdf_streams(xdf_path, out_dir)
    
    # Optional: create simple analysis
    create_simple_plot(xdf_path)
    
    print("\n✓ Processing complete!")

Loading C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf ...
Loaded 2 streams using pyxdf

--- streams detected ---
Audio: Audio_P001_20250506_171059
Mouse: MouseClicks_P001_20250506_171059
✓ Saved combined data -> C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT\Audio_P001_20250506_171059_combined_data.csv
  - Audio samples: 77793280, sr=44100


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [3]:
#!/usr/bin/env python3
"""
PPS Experiment – XDF Stream Extractor (Dependency-Free Version)

This script loads a LabRecorder .xdf file, finds the Audio and MouseEvents streams,
and saves them as WAV and CSV files without relying on pandas, soundfile, or matplotlib.

Usage:
    python xdf_extractor.py
    
Or modify the paths in the USER CONFIG section below.
"""

import os
import pathlib
import re
import json
import struct
import csv
from typing import Optional, List, Dict, Any
import xml.etree.ElementTree as ET

# Try to import numpy, with fallback
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    print("Warning: NumPy not available. Using pure Python fallbacks.")
    HAS_NUMPY = False

def load_xdf_partial(filepath: str, max_audio_duration: float = 300.0) -> tuple:
    """
    Load XDF file but only decode the first few minutes of audio data.
    This avoids the computational expense of decoding the entire audio stream.
    
    Returns: (streams_with_limited_audio, header)
    """
    try:
        import pyxdf
        print(f"Loading XDF with pyxdf (limiting audio to first {max_audio_duration/60:.1f} minutes)...")
        
        # Load the XDF file normally
        streams, header = pyxdf.load_xdf(filepath)
        
        # Limit audio streams to first N minutes
        for stream in streams:
            if stream['info'].get('type', [''])[0] == 'Audio':
                sample_rate = int(float(stream['info'].get('nominal_srate', ['44100'])[0]))
                max_samples = int(max_audio_duration * sample_rate)
                
                original_samples = len(stream['time_series'])
                if original_samples > max_samples:
                    print(f"Limiting audio from {original_samples} to {max_samples} samples ({max_audio_duration/60:.1f} min)")
                    stream['time_series'] = stream['time_series'][:max_samples]
                    stream['time_stamps'] = stream['time_stamps'][:max_samples]
                else:
                    print(f"Audio stream has {original_samples} samples (< {max_samples} limit)")
        
        return streams, header
        
    except ImportError:
        print("pyxdf not available, using simple XDF loader with audio limiting...")
        return read_xdf_simple_limited(filepath, max_audio_duration)
    except Exception as e:
        print(f"pyxdf failed: {e}, falling back to simple loader...")
        return read_xdf_simple_limited(filepath, max_audio_duration)

def read_xdf_simple_limited(filepath: str, max_audio_duration: float = 300.0) -> tuple:
    """
    Simplified XDF reader that only loads the first few minutes of audio data.
    """
    streams = []
    audio_sample_limit = {}  # stream_id -> max_samples
    
    with open(filepath, 'rb') as f:
        header = {}
        current_stream_id = -1
        
        while True:
            try:
                length_bytes = f.read(4)
                if len(length_bytes) < 4:
                    break
                length = struct.unpack('<I', length_bytes)[0]
                
                if length == 0:
                    continue
                    
                tag = struct.unpack('<H', f.read(2))[0]
                payload = f.read(length - 2)
                
                if tag == 2:  # Stream header
                    stream_info = parse_stream_header(payload)
                    stream = {
                        'info': stream_info,
                        'time_series': [],
                        'time_stamps': []
                    }
                    streams.append(stream)
                    current_stream_id = len(streams) - 1
                    
                    # Set sample limit for audio streams
                    if stream_info.get('type', [''])[0] == 'Audio':
                        sample_rate = int(stream_info.get('nominal_srate', ['44100'])[0])
                        audio_sample_limit[current_stream_id] = int(max_audio_duration * sample_rate)
                        print(f"Audio stream detected: limiting to {audio_sample_limit[current_stream_id]} samples")
                    
                elif tag == 3:  # Samples
                    if streams and current_stream_id >= 0:
                        # Check if this is an audio stream with limits
                        if current_stream_id in audio_sample_limit:
                            current_samples = len(streams[current_stream_id]['time_series'])
                            if current_samples >= audio_sample_limit[current_stream_id]:
                                # Skip this audio sample chunk - we have enough
                                continue
                        
                        parse_samples_limited(payload, streams[current_stream_id], 
                                            audio_sample_limit.get(current_stream_id))
                        
                elif tag == 6:  # Stream footer
                    continue
                    
            except struct.error:
                break
                
    return streams, header

def parse_samples_limited(payload: bytes, stream: dict, sample_limit: Optional[int] = None) -> None:
    """Parse sample data with optional sample count limiting for audio streams."""
    try:
        channel_count = int(stream['info'].get('channel_count', ['1'])[0])
        sample_format = stream['info'].get('channel_format', ['float32'])[0]
        
        # Simple parsing - assumes float32 format
        if sample_format == 'float32':
            sample_size = 4
        else:
            sample_size = 4  # Default
            
        # Each sample has timestamp (8 bytes) + channel data
        timestamp_size = 8
        record_size = timestamp_size + (channel_count * sample_size)
        
        offset = 0
        samples_added = 0
        current_sample_count = len(stream['time_series'])
        
        while offset + record_size <= len(payload):
            # Check sample limit for audio streams
            if sample_limit is not None and current_sample_count + samples_added >= sample_limit:
                break
                
            # Read timestamp
            timestamp = struct.unpack('<d', payload[offset:offset+8])[0]
            offset += 8
            
            # Read channel data
            sample_data = []
            for _ in range(channel_count):
                if sample_format == 'float32':
                    value = struct.unpack('<f', payload[offset:offset+4])[0]
                else:
                    value = 0.0
                sample_data.append(value)
                offset += 4
                
            stream['time_stamps'].append(timestamp)
            stream['time_series'].append(sample_data)
            samples_added += 1
            
    except:
        pass  # Skip malformed samples

def parse_stream_header(payload: bytes) -> dict:
    """Parse stream header XML."""
    try:
        xml_str = payload.decode('utf-8')
        root = ET.fromstring(xml_str)
        
        info = {}
        for elem in root:
            if len(list(elem)) == 0:  # Leaf node
                info[elem.tag] = [elem.text or '']
            else:  # Has children
                info[elem.tag] = {}
                for child in elem:
                    info[elem.tag][child.tag] = [child.text or '']
                    
        return info
    except:
        return {'name': ['Unknown'], 'type': ['Unknown'], 'channel_count': ['1']}

def parse_samples(payload: bytes, stream: dict) -> None:
    """Parse sample data from payload."""
    try:
        channel_count = int(stream['info'].get('channel_count', ['1'])[0])
        sample_format = stream['info'].get('channel_format', ['float32'])[0]
        
        # Simple parsing - assumes float32 format
        if sample_format == 'float32':
            sample_size = 4
        else:
            sample_size = 4  # Default
            
        # Each sample has timestamp (8 bytes) + channel data
        timestamp_size = 8
        record_size = timestamp_size + (channel_count * sample_size)
        
        offset = 0
        while offset + record_size <= len(payload):
            # Read timestamp
            timestamp = struct.unpack('<d', payload[offset:offset+8])[0]
            offset += 8
            
            # Read channel data
            sample_data = []
            for _ in range(channel_count):
                if sample_format == 'float32':
                    value = struct.unpack('<f', payload[offset:offset+4])[0]
                else:
                    value = 0.0
                sample_data.append(value)
                offset += 4
                
            stream['time_stamps'].append(timestamp)
            stream['time_series'].append(sample_data)
            
    except:
        pass  # Skip malformed samples

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def find_stream(streams: List[Dict], *, type_=None, name_regex=None, channel_count=None) -> Optional[Dict]:
    """Return the first stream that matches all given criteria."""
    for s in streams:
        info = s['info']
        name = info.get('name', [''])[0]
        stype = info.get('type', [''])[0]
        nch = int(info.get('channel_count', ['1'])[0])
        
        if type_ is not None and stype != type_:
            continue
        if channel_count is not None and nch != channel_count:
            continue
        if name_regex is not None and re.fullmatch(name_regex, name) is None:
            continue
        return s
    return None

def detect_calibration_end(audio_data: List[List[float]], sample_rate: int, max_duration: float = 300.0) -> float:
    """
    Detect the end of the 5-burst calibration beep sequence.
    
    Calibration pattern:
    - 5 bursts of 1 kHz tone (20ms each) + 80ms silence
    - Total duration: ~500ms
    - Both channels identical during calibration
    
    Returns: time offset (seconds) where calibration ends, or 0 if not found
    """
    if HAS_NUMPY:
        samples = np.array(audio_data, dtype=np.float32)
        if samples.ndim == 1:
            samples = samples.reshape(-1, 1)
        n_frames, n_channels = samples.shape
    else:
        samples = audio_data
        n_frames = len(samples)
        n_channels = len(samples[0]) if n_frames > 0 else 1
    
    # Limit to first 5 minutes for efficiency
    max_samples = int(max_duration * sample_rate)
    search_frames = min(n_frames, max_samples)
    
    print(f"Searching for calibration in first {search_frames/sample_rate:.1f}s of audio...")
    
    # Expected calibration parameters
    burst_duration = 0.02  # 20ms tone
    silence_duration = 0.08  # 80ms silence
    total_burst_cycle = burst_duration + silence_duration  # 100ms
    expected_bursts = 5
    expected_calib_duration = total_burst_cycle * expected_bursts  # 500ms
    
    # Convert to samples
    burst_samples = int(burst_duration * sample_rate)
    silence_samples = int(silence_duration * sample_rate)
    cycle_samples = burst_samples + silence_samples
    
    if HAS_NUMPY:
        # Use FFT to detect 1 kHz content in sliding windows
        window_size = burst_samples
        step_size = window_size // 4
        
        # Target frequency bin for 1 kHz
        freqs = np.fft.fftfreq(window_size, 1/sample_rate)
        target_bin = np.argmin(np.abs(freqs - 1000))  # Closest to 1 kHz
        
        # Scan for burst pattern
        burst_detections = []
        
        for start_idx in range(0, min(search_frames - window_size, int(2 * sample_rate)), step_size):
            # Analyze window (use mono signal)
            if n_channels > 1:
                window = samples[start_idx:start_idx + window_size, 0]
            else:
                window = samples[start_idx:start_idx + window_size, 0]
            
            # FFT magnitude at 1 kHz
            fft_mag = np.abs(np.fft.fft(window))
            tone_strength = fft_mag[target_bin]
            
            # Also check RMS energy
            rms_energy = np.sqrt(np.mean(window**2))
            
            # Detection criteria: strong 1 kHz + sufficient energy
            if tone_strength > 100 and rms_energy > 0.01:  # Thresholds may need tuning
                burst_detections.append(start_idx / sample_rate)
        
        if len(burst_detections) >= 3:  # Need at least 3 bursts detected
            # Find the end of calibration pattern
            # Look for where channels start to diverge (looming vs tactile)
            if n_channels >= 2:
                # Search after expected calibration duration
                search_start = int(expected_calib_duration * sample_rate)
                search_end = min(search_frames, search_start + int(1.0 * sample_rate))  # Search 1s after expected end
                
                for i in range(search_start, search_end, sample_rate // 100):  # Check every 10ms
                    if i + sample_rate // 20 < n_frames:  # 50ms window
                        window_left = samples[i:i + sample_rate // 20, 0]
                        window_right = samples[i:i + sample_rate // 20, 1]
                        
                        # Calculate correlation between channels
                        correlation = np.corrcoef(window_left, window_right)[0, 1]
                        
                        # When correlation drops significantly, calibration has ended
                        if correlation < 0.8:  # Channels are no longer identical
                            calib_end_time = i / sample_rate
                            print(f"✓ Calibration end detected at {calib_end_time:.3f}s (correlation: {correlation:.3f})")
                            return calib_end_time
            
            # Fallback: use expected duration
            calib_end_time = expected_calib_duration
            print(f"✓ Using expected calibration duration: {calib_end_time:.3f}s")
            return calib_end_time
        else:
            print(f"⚠️  Only {len(burst_detections)} bursts detected, using expected duration")
            return expected_calib_duration
    
    else:  # Pure Python fallback
        # Simple energy-based detection
        window_size = int(0.02 * sample_rate)  # 20ms windows
        high_energy_periods = []
        
        for i in range(0, min(search_frames - window_size, int(2 * sample_rate)), window_size):
            if n_channels > 1:
                window_energy = sum(abs(samples[j][0]) for j in range(i, i + window_size)) / window_size
            else:
                window_energy = sum(abs(samples[j][0]) for j in range(i, i + window_size)) / window_size
            
            if window_energy > 0.05:  # Threshold for significant energy
                high_energy_periods.append(i / sample_rate)
        
        if len(high_energy_periods) >= 3:
            calib_end_time = expected_calib_duration
            print(f"✓ Energy-based detection: using expected duration {calib_end_time:.3f}s")
            return calib_end_time
    
    # Default fallback
    print("⚠️  Could not detect calibration sequence, using 0.5s default")
    return 0.5

def save_calibration_analysis_csv(filename: str, audio_data: List[List[float]], audio_timestamps: List[float], 
                                 sample_rate: int, mouse_data: List[List[float]], mouse_timestamps: List[float]) -> None:
    """
    Detect the end of the 5-burst calibration beep sequence.
    
    Calibration pattern:
    - 5 bursts of 1 kHz tone (20ms each) + 80ms silence
    - Total duration: ~500ms
    - Both channels identical during calibration
    
    Returns: time offset (seconds) where calibration ends, or 0 if not found
    """
    if HAS_NUMPY:
        samples = np.array(audio_data, dtype=np.float32)
        if samples.ndim == 1:
            samples = samples.reshape(-1, 1)
        n_frames, n_channels = samples.shape
    else:
        samples = audio_data
        n_frames = len(samples)
        n_channels = len(samples[0]) if n_frames > 0 else 1
    
    # Limit to first 5 minutes for efficiency
    max_samples = int(max_duration * sample_rate)
    search_frames = min(n_frames, max_samples)
    
    print(f"Searching for calibration in first {search_frames/sample_rate:.1f}s of audio...")
    
    # Expected calibration parameters
    burst_duration = 0.02  # 20ms tone
    silence_duration = 0.08  # 80ms silence
    total_burst_cycle = burst_duration + silence_duration  # 100ms
    expected_bursts = 5
    expected_calib_duration = total_burst_cycle * expected_bursts  # 500ms
    
    # Convert to samples
    burst_samples = int(burst_duration * sample_rate)
    silence_samples = int(silence_duration * sample_rate)
    cycle_samples = burst_samples + silence_samples
    
    if HAS_NUMPY:
        # Use FFT to detect 1 kHz content in sliding windows
        window_size = burst_samples
        step_size = window_size // 4
        
        # Target frequency bin for 1 kHz
        freqs = np.fft.fftfreq(window_size, 1/sample_rate)
        target_bin = np.argmin(np.abs(freqs - 1000))  # Closest to 1 kHz
        
        # Scan for burst pattern
        burst_detections = []
        
        for start_idx in range(0, min(search_frames - window_size, int(2 * sample_rate)), step_size):
            # Analyze window (use mono signal)
            if n_channels > 1:
                window = samples[start_idx:start_idx + window_size, 0]
            else:
                window = samples[start_idx:start_idx + window_size, 0]
            
            # FFT magnitude at 1 kHz
            fft_mag = np.abs(np.fft.fft(window))
            tone_strength = fft_mag[target_bin]
            
            # Also check RMS energy
            rms_energy = np.sqrt(np.mean(window**2))
            
            # Detection criteria: strong 1 kHz + sufficient energy
            if tone_strength > 100 and rms_energy > 0.01:  # Thresholds may need tuning
                burst_detections.append(start_idx / sample_rate)
        
        if len(burst_detections) >= 3:  # Need at least 3 bursts detected
            # Find the end of calibration pattern
            # Look for where channels start to diverge (looming vs tactile)
            if n_channels >= 2:
                # Search after expected calibration duration
                search_start = int(expected_calib_duration * sample_rate)
                search_end = min(search_frames, search_start + int(1.0 * sample_rate))  # Search 1s after expected end
                
                for i in range(search_start, search_end, sample_rate // 100):  # Check every 10ms
                    if i + sample_rate // 20 < n_frames:  # 50ms window
                        window_left = samples[i:i + sample_rate // 20, 0]
                        window_right = samples[i:i + sample_rate // 20, 1]
                        
                        # Calculate correlation between channels
                        correlation = np.corrcoef(window_left, window_right)[0, 1]
                        
                        # When correlation drops significantly, calibration has ended
                        if correlation < 0.8:  # Channels are no longer identical
                            calib_end_time = i / sample_rate
                            print(f"✓ Calibration end detected at {calib_end_time:.3f}s (correlation: {correlation:.3f})")
                            return calib_end_time
            
            # Fallback: use expected duration
            calib_end_time = expected_calib_duration
            print(f"✓ Using expected calibration duration: {calib_end_time:.3f}s")
            return calib_end_time
        else:
            print(f"⚠️  Only {len(burst_detections)} bursts detected, using expected duration")
            return expected_calib_duration
    
    else:  # Pure Python fallback
        # Simple energy-based detection
        window_size = int(0.02 * sample_rate)  # 20ms windows
        high_energy_periods = []
        
        for i in range(0, min(search_frames - window_size, int(2 * sample_rate)), window_size):
            if n_channels > 1:
                window_energy = sum(abs(samples[j][0]) for j in range(i, i + window_size)) / window_size
            else:
                window_energy = sum(abs(samples[j][0]) for j in range(i, i + window_size)) / window_size
            
            if window_energy > 0.05:  # Threshold for significant energy
                high_energy_periods.append(i / sample_rate)
        
        if len(high_energy_periods) >= 3:
            calib_end_time = expected_calib_duration
            print(f"✓ Energy-based detection: using expected duration {calib_end_time:.3f}s")
            return calib_end_time
    
    # Default fallback
    print("⚠️  Could not detect calibration sequence, using 0.5s default")
    return 0.5
def save_calibration_analysis_csv(filename: str, audio_data: List[List[float]], audio_timestamps: List[float], 
                                 sample_rate: int, mouse_data: List[List[float]], mouse_timestamps: List[float]) -> None:
    """
    Save combined audio and mouse data with calibration-corrected timestamps.
    Only processes first 5 minutes of audio to save time, focuses on calibration detection.
    """
    
    # Detect calibration end point
    calib_end_time = detect_calibration_end(audio_data, sample_rate, max_duration=300.0)
    
    # Process only a subset of audio data for efficiency (first 5 minutes or until calibration + 30s)
    analysis_duration = max(calib_end_time + 30.0, 60.0)  # At least 1 minute, or calib + 30s
    max_audio_samples = int(analysis_duration * sample_rate)
    
    if HAS_NUMPY:
        samples = np.array(audio_data[:max_audio_samples], dtype=np.float32) if len(audio_data) > max_audio_samples else np.array(audio_data, dtype=np.float32)
        if samples.ndim == 1:
            samples = samples.reshape(-1, 1)
        n_frames = samples.shape[0]
        n_channels = samples.shape[1]
    else:
        samples = audio_data[:max_audio_samples] if len(audio_data) > max_audio_samples else audio_data
        if len(samples) > 0 and isinstance(samples[0], (int, float)):
            samples = [[x] for x in samples]
        n_frames = len(samples)
        n_channels = len(samples[0]) if n_frames > 0 else 1
    
    print(f"Processing {n_frames/sample_rate:.1f}s of audio data (reduced from full recording)")
    
    # Create corrected time axis - subtract calibration end time to make it the new zero
    if audio_timestamps is not None and len(audio_timestamps) >= n_frames:
        # Use first timestamp as reference
        original_t0 = audio_timestamps[0]
        # Calibration end in absolute time
        calib_end_absolute = original_t0 + calib_end_time
        # Create corrected time axis where calibration end = 0
        corrected_time_axis = [(audio_timestamps[i] - calib_end_absolute) for i in range(n_frames)]
    else:
        # Generate time axis based on sample rate, corrected for calibration
        corrected_time_axis = [(i / sample_rate) - calib_end_time for i in range(n_frames)]
        calib_end_absolute = calib_end_time  # For mouse timestamp correction
        original_t0 = 0
    
    # Process mouse clicks with corrected timestamps
    mouse_events = {}  # corrected_time -> (x, y)
    corrected_mouse_times = []
    
    if mouse_data is not None and mouse_timestamps is not None and len(mouse_data) > 0:
        for i, (mouse_time, mouse_pos) in enumerate(zip(mouse_timestamps, mouse_data)):
            # Correct mouse timestamp relative to calibration end
            if audio_timestamps is not None and len(audio_timestamps) > 0:
                corrected_mouse_time = mouse_time - calib_end_absolute
            else:
                corrected_mouse_time = mouse_time - calib_end_time
            
            corrected_mouse_times.append(corrected_mouse_time)
            
            # Find nearest audio sample time for alignment (only if within our analysis window)
            if corrected_mouse_time >= corrected_time_axis[0] and corrected_mouse_time <= corrected_time_axis[-1]:
                closest_audio_idx = min(range(len(corrected_time_axis)), 
                                      key=lambda j: abs(corrected_time_axis[j] - corrected_mouse_time))
                aligned_time = corrected_time_axis[closest_audio_idx]
                mouse_events[aligned_time] = (mouse_pos[1], mouse_pos[2])  # x, y coordinates
    
    # Create CSV with analysis data
    with open(filename, 'w', newline='') as csvfile:
        # Headers based on audio channels
        if n_channels == 1:
            headers = ['corrected_time_sec', 'audio_amplitude', 'mouse_click', 'mouse_x', 'mouse_y', 'calibration_phase']
        elif n_channels == 2:
            headers = ['corrected_time_sec', 'audio_left', 'audio_right', 'mouse_click', 'mouse_x', 'mouse_y', 'calibration_phase']
        else:
            headers = ['corrected_time_sec'] + [f'audio_ch{j+1}' for j in range(n_channels)] + ['mouse_click', 'mouse_x', 'mouse_y', 'calibration_phase']
        
        writer = csv.writer(csvfile)
        writer.writerow(headers)
        
        # Write data with corrected timestamps
        for i in range(n_frames):
            corrected_time = corrected_time_axis[i]
            row = [corrected_time]
            
            # Add audio data
            for j in range(n_channels):
                if HAS_NUMPY:
                    amplitude = float(samples[i, j])
                else:
                    amplitude = samples[i][j] if i < len(samples) else 0.0
                row.append(amplitude)
            
            # Add mouse data (if click occurred at this time)
            if corrected_time in mouse_events:
                row.extend([1, mouse_events[corrected_time][0], mouse_events[corrected_time][1]])
            else:
                row.extend([0, '', ''])
            
            # Add calibration phase indicator
            if corrected_time < 0:
                row.append('calibration')
            elif corrected_time < 1.0:
                row.append('post_calibration')
            else:
                row.append('experiment')
            
            writer.writerow(row)
    
    # Save summary of corrected mouse clicks
    if corrected_mouse_times:
        summary_file = filename.replace('.csv', '_mouse_summary.csv')
        with open(summary_file, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['original_timestamp', 'corrected_timestamp', 'x_px', 'y_px'])
            
            for i, (orig_time, corr_time, mouse_pos) in enumerate(zip(mouse_timestamps, corrected_mouse_times, mouse_data)):
                writer.writerow([orig_time, corr_time, mouse_pos[1], mouse_pos[2]])
        
        print(f"✓ Mouse click summary saved to: {summary_file}")
    
    print(f"✓ Calibration analysis complete:")
    print(f"  - Calibration end detected at: {calib_end_time:.3f}s")
    print(f"  - Mouse clicks corrected: {len(corrected_mouse_times) if corrected_mouse_times else 0}")
    print(f"  - New timestamp 0 = original timestamp {calib_end_time:.3f}s")
    
    # Process audio data
    if HAS_NUMPY:
        audio_samples = np.array(audio_data, dtype=np.float32)
        if audio_samples.ndim == 1:
            audio_samples = audio_samples.reshape(-1, 1)
        n_audio_frames = audio_samples.shape[0]
        n_channels = audio_samples.shape[1]
    else:
        audio_samples = audio_data
        if len(audio_data) > 0 and isinstance(audio_data[0], (int, float)):
            audio_samples = [[x] for x in audio_data]
        n_audio_frames = len(audio_samples)
        n_channels = len(audio_samples[0]) if len(audio_samples) > 0 else 1
    
    # Create synchronized time axis from audio (this is our master timeline)
    if audio_timestamps is not None and len(audio_timestamps) == n_audio_frames:
        t0 = audio_timestamps[0]  # Reference time
        audio_time_axis = [t - t0 for t in audio_timestamps]
    else:
        audio_time_axis = [i / sample_rate for i in range(n_audio_frames)]
        t0 = 0
    
    # Process mouse clicks - convert to relative time and create lookup
    mouse_events = {}  # time -> (x, y)
    if mouse_data is not None and mouse_timestamps is not None and len(mouse_data) > 0:
        for i, (mouse_time, mouse_pos) in enumerate(zip(mouse_timestamps, mouse_data)):
            rel_time = mouse_time - t0
            # Round to nearest audio sample time for alignment
            closest_audio_idx = min(range(len(audio_time_axis)), 
                                  key=lambda j: abs(audio_time_axis[j] - rel_time))
            aligned_time = audio_time_axis[closest_audio_idx]
            mouse_events[aligned_time] = (mouse_pos[1], mouse_pos[2])  # x, y coordinates
    
    # Create CSV with combined data
    with open(filename, 'w', newline='') as csvfile:
        # Prepare headers based on audio channels
        if n_channels == 1:
            headers = ['time_sec', 'audio_amplitude', 'mouse_click', 'mouse_x', 'mouse_y']
        elif n_channels == 2:
            headers = ['time_sec', 'audio_left', 'audio_right', 'mouse_click', 'mouse_x', 'mouse_y']
        else:
            headers = ['time_sec'] + [f'audio_ch{j+1}' for j in range(n_channels)] + ['mouse_click', 'mouse_x', 'mouse_y']
        
        writer = csv.writer(csvfile)
        writer.writerow(headers)
        
        # Write synchronized data
        for i in range(n_audio_frames):
            time_point = audio_time_axis[i]
            row = [time_point]
            
            # Add audio data
            for j in range(n_channels):
                if HAS_NUMPY:
                    amplitude = float(audio_samples[i, j])
                else:
                    amplitude = audio_samples[i][j] if i < len(audio_samples) else 0.0
                row.append(amplitude)
            
            # Add mouse data (if click occurred at this time)
            if time_point in mouse_events:
                row.extend([1, mouse_events[time_point][0], mouse_events[time_point][1]])  # click=1, x, y
            else:
                row.extend([0, '', ''])  # no click, empty coordinates
            
            writer.writerow(row)

def save_csv_simple(filename: str, data: List[List[float]], headers: List[str]) -> None:
    """Save data as CSV file without pandas dependency."""
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)
        for row in data:
            writer.writerow(row)

# ============================================================================
# MAIN EXTRACTION LOGIC
# ============================================================================

def extract_calibration_and_timestamps(xdf_path: str, out_dir: str, max_audio_minutes: float = 5.0) -> None:
    """
    Fast extraction: only decode first few minutes of audio to find calibration,
    then correct all mouse timestamps without processing full audio stream.
    """
    out_dir = pathlib.Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Loading {xdf_path} (limiting audio to first {max_audio_minutes} minutes)...")
    
    # Load XDF with limited audio
    streams, _ = load_xdf_partial(xdf_path, max_audio_minutes * 60)
    
    if not streams:
        print("No streams found in XDF file!")
        return
    
    # Identify streams
    audio_stream = (
        find_stream(streams, type_="Audio", channel_count=2)
        or find_stream(streams, name_regex=r".*_Audio_.*")
    )
    
    mouse_stream = (
        find_stream(streams, type_="MouseEvents", channel_count=3)
        or find_stream(streams, name_regex=r".*_MouseClicks_.*")
    )
    
    print("\n--- streams detected ---")
    print("Audio:", audio_stream['info'].get('name', ['not found'])[0] if audio_stream else "not found")
    print("Mouse:", mouse_stream['info'].get('name', ['not found'])[0] if mouse_stream else "not found")
    
    if not audio_stream:
        print("❌ No audio stream found - cannot detect calibration!")
        return
        
    if not mouse_stream:
        print("❌ No mouse stream found - nothing to correct!")
        return
    
    # Process calibration detection on limited audio
    audio_name = audio_stream["info"]["name"][0]
    sr_str = audio_stream["info"].get("nominal_srate", ["44100"])[0]
    sr = int(float(sr_str))
    
    limited_audio_data = audio_stream["time_series"]
    limited_audio_timestamps = audio_stream["time_stamps"]
    
    print(f"\n--- Calibration Detection ---")
    print(f"Audio samples to analyze: {len(limited_audio_data)} ({len(limited_audio_data)/sr:.1f}s)")
    
    # Detect calibration end
    calib_end_time = detect_calibration_end(limited_audio_data, sr)
    
    # Get ALL mouse data (this is small, no need to limit)
    all_mouse_data = mouse_stream['time_series']
    all_mouse_timestamps = mouse_stream['time_stamps']
    
    # Calculate calibration end in absolute time
    if limited_audio_timestamps and len(limited_audio_timestamps) > 0:
        audio_start_absolute = limited_audio_timestamps[0]
        calib_end_absolute = audio_start_absolute + calib_end_time
    else:
        calib_end_absolute = calib_end_time
        audio_start_absolute = 0
    
    # Correct ALL mouse timestamps
    corrected_mouse_data = []
    for i, (mouse_time, mouse_pos) in enumerate(zip(all_mouse_timestamps, all_mouse_data)):
        corrected_time = mouse_time - calib_end_absolute
        corrected_mouse_data.append({
            'original_timestamp': mouse_time,
            'corrected_timestamp': corrected_time,
            'x_px': mouse_pos[1] if len(mouse_pos) > 1 else 0,
            'y_px': mouse_pos[2] if len(mouse_pos) > 2 else 0,
            'phase': 'calibration' if corrected_time < 0 else ('early' if corrected_time < 5 else 'experiment')
        })
    
    # Save corrected timestamps CSV
    timestamps_file = out_dir / f"{audio_name}_corrected_timestamps.csv"
    with open(timestamps_file, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['original_timestamp', 'corrected_timestamp', 'x_px', 'y_px', 'phase'])
        
        for entry in corrected_mouse_data:
            writer.writerow([
                entry['original_timestamp'],
                entry['corrected_timestamp'], 
                entry['x_px'],
                entry['y_px'],
                entry['phase']
            ])
    
    # Create summary file
    summary_file = out_dir / f"{audio_name}_calibration_summary.txt"
    with open(summary_file, 'w') as f:
        f.write(f"Calibration Analysis Summary\n")
        f.write(f"===========================\n\n")
        f.write(f"XDF File: {pathlib.Path(xdf_path).name}\n")
        f.write(f"Audio Stream: {audio_name}\n")
        f.write(f"Sample Rate: {sr} Hz\n")
        f.write(f"Audio Analyzed: {len(limited_audio_data)/sr:.1f}s (limited from full recording)\n\n")
        f.write(f"CALIBRATION DETECTION:\n")
        f.write(f"- Calibration sequence ends at: {calib_end_time:.3f}s\n")
        f.write(f"- Original audio start: {audio_start_absolute:.6f}\n")
        f.write(f"- Calibration end (absolute): {calib_end_absolute:.6f}\n\n")
        f.write(f"MOUSE CLICK CORRECTION:\n")
        f.write(f"- Total mouse clicks: {len(corrected_mouse_data)}\n")
        
        calib_clicks = sum(1 for entry in corrected_mouse_data if entry['phase'] == 'calibration')
        early_clicks = sum(1 for entry in corrected_mouse_data if entry['phase'] == 'early')
        exp_clicks = sum(1 for entry in corrected_mouse_data if entry['phase'] == 'experiment')
        
        f.write(f"- During calibration (t < 0): {calib_clicks}\n")
        f.write(f"- First 5s after calibration (0 ≤ t < 5): {early_clicks}\n")
        f.write(f"- Rest of experiment (t ≥ 5): {exp_clicks}\n\n")
        f.write(f"TIMESTAMP ZERO POINT:\n")
        f.write(f"- New t=0 corresponds to original timestamp: {calib_end_absolute:.6f}\n")
        f.write(f"- This is {calib_end_time:.3f}s after audio recording started\n")
    
    # Print results
    print(f"\n✅ CALIBRATION ANALYSIS COMPLETE!")
    print(f"📁 Saved corrected timestamps: {timestamps_file}")
    print(f"📄 Saved summary: {summary_file}")
    print(f"\n📊 RESULTS:")
    print(f"   - Calibration ends at: {calib_end_time:.3f}s")
    print(f"   - Mouse clicks corrected: {len(corrected_mouse_data)}")
    print(f"   - Clicks during calibration: {calib_clicks}")
    print(f"   - Clicks in first 5s of experiment: {early_clicks}")
    print(f"   - Total experiment clicks: {exp_clicks}")
    print(f"\n⏰ New timestamp 0 = original {calib_end_absolute:.6f}")
    print(f"   (saves {calib_end_time:.3f}s of calibration beep time)")


# ============================================================================
# SIMPLE VISUALIZATION (optional, without matplotlib)
# ============================================================================

def create_calibration_analysis(xdf_path: str) -> None:
    """Create a calibration-focused analysis of the data."""
    print("\n--- Calibration Analysis ---")
    
    try:
        import pyxdf
        streams, _ = pyxdf.load_xdf(xdf_path)
    except:
        streams, _ = read_xdf_simple(xdf_path)
    
    audio = find_stream(streams, type_="Audio")
    mouse = find_stream(streams, type_="MouseEvents")
    
    if audio and len(audio["time_series"]) > 0:
        sr = int(float(audio["info"].get("nominal_srate", ["44100"])[0]))
        
        # Analyze first 5 minutes only
        max_samples = min(len(audio["time_series"]), int(300 * sr))
        sample_duration = max_samples / sr
        
        print(f"Audio stream: {len(audio['time_series'])} total samples ({len(audio['time_series'])/sr:.1f}s)")
        print(f"Analyzing first: {max_samples} samples ({sample_duration:.1f}s)")
        
        # Detect calibration in the subset
        calib_end = detect_calibration_end(audio["time_series"][:max_samples], sr)
        print(f"Calibration sequence ends at: {calib_end:.3f}s")
    
    if mouse and len(mouse["time_series"]) > 0:
        mouse_times = mouse["time_stamps"] if HAS_NUMPY else [t for t in mouse["time_stamps"]]
        if audio:
            audio_start = audio["time_stamps"][0] if audio["time_stamps"] else 0
            calib_end_absolute = audio_start + calib_end
            corrected_times = [t - calib_end_absolute for t in mouse_times]
            
            print(f"Mouse clicks: {len(mouse['time_series'])}")
            print(f"First 5 clicks (corrected times): {corrected_times[:5]}")
            
            # Count clicks in different phases
            calib_clicks = sum(1 for t in corrected_times if t < 0)
            early_clicks = sum(1 for t in corrected_times if 0 <= t < 5)
            total_clicks = len(corrected_times)
            
            print(f"Click distribution:")
            print(f"  - During calibration (t < 0): {calib_clicks}")
            print(f"  - First 5s of experiment (0 ≤ t < 5): {early_clicks}")
            print(f"  - Total clicks: {total_clicks}")
        else:
            print(f"Mouse: {len(mouse['time_series'])} clicks (no audio reference)")
    
    print("Analysis complete!")

# ============================================================================
# USER CONFIGURATION & MAIN
# ============================================================================

if __name__ == "__main__":
    # ---- USER INPUTS -------------------------------------------------------
    xdf_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf"
    out_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT"
    # -----------------------------------------------------------------------
    
    # Check if file exists
    if not os.path.exists(xdf_path):
        print(f"Error: XDF file not found: {xdf_path}")
        print("Please update the xdf_path variable in the script.")
        exit(1)
    
    # Extract streams
    extract_xdf_streams(xdf_path, out_dir)
    
    # Calibration analysis
    create_calibration_analysis(xdf_path)
    
    print("\n✓ Processing complete!")

Loading C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf ...
Loaded 2 streams using pyxdf

--- streams detected ---
Audio: Audio_P001_20250506_171059
Mouse: MouseClicks_P001_20250506_171059
✓ Saved combined data -> C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT\Audio_P001_20250506_171059_combined_data.csv
  - Audio samples: 77793280, sr=44100


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [4]:
#!/usr/bin/env python3
"""
XDF Calibration Detector - Ultra-Efficient Streaming Version

Processes XDF files chunk by chunk, stopping as soon as the calibration beep
sequence is detected. Only decodes the minimum amount of audio data necessary.

Key Features:
- Streaming XDF processing (chunk by chunk)
- Stops immediately after detecting calibration end
- Corrects all mouse timestamps without full audio decode
- Handles 5-burst 1kHz calibration sequence (20ms tone + 80ms silence × 5)
"""

import os
import pathlib
import re
import struct
import csv
from typing import Optional, List, Dict, Any, Tuple
import xml.etree.ElementTree as ET

# Try to import numpy, with fallback
try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    print("Warning: NumPy not available. Using pure Python fallbacks.")
    HAS_NUMPY = False

# ============================================================================
# STREAMING XDF PROCESSOR
# ============================================================================

class StreamingXDFProcessor:
    """
    Processes XDF files chunk by chunk, stopping when calibration is detected.
    """
    
    def __init__(self, filepath: str):
        self.filepath = filepath
        self.file_handle = None
        self.streams_info = {}  # stream_id -> info dict
        self.audio_stream_id = None
        self.mouse_stream_id = None
        self.audio_chunks = []  # List of (timestamp, samples) tuples
        self.all_mouse_data = []  # Will collect all mouse data
        self.sample_rate = 44100
        self.calibration_detected = False
        self.calibration_end_time = None
        self.audio_start_timestamp = None
        
    def __enter__(self):
        print("🔧 Opening XDF file...")
        self.file_handle = open(self.filepath, 'rb')
        print(f"✅ File opened successfully: {pathlib.Path(self.filepath).name}")
        return self
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file_handle:
            print("🔧 Closing XDF file...")
            self.file_handle.close()
            print("✅ File closed successfully")
    
    def parse_stream_header(self, payload: bytes) -> dict:
        """Parse stream header XML."""
        print("  🔍 Parsing stream header...")
        try:
            xml_str = payload.decode('utf-8')
            root = ET.fromstring(xml_str)
            
            info = {}
            for elem in root:
                if len(list(elem)) == 0:  # Leaf node
                    info[elem.tag] = [elem.text or '']
                else:  # Has children
                    info[elem.tag] = {}
                    for child in elem:
                        info[elem.tag][child.tag] = [child.text or '']
            
            stream_name = info.get('name', ['Unknown'])[0]
            stream_type = info.get('type', ['Unknown'])[0]
            print(f"  ✅ Stream header parsed: {stream_name} (type: {stream_type})")
            return info
        except Exception as e:
            print(f"  ⚠️ Failed to parse stream header: {e}")
            return {'name': ['Unknown'], 'type': ['Unknown'], 'channel_count': ['1']}
    
    def parse_audio_samples(self, payload: bytes, channel_count: int) -> List[Tuple[float, List[float]]]:
        """Parse audio sample data from payload."""
        print(f"  🎵 Parsing audio samples (payload size: {len(payload)} bytes, {channel_count} channels)...")
        samples = []
        try:
            sample_size = 4  # float32
            timestamp_size = 8
            record_size = timestamp_size + (channel_count * sample_size)
            expected_samples = len(payload) // record_size
            print(f"    Expected {expected_samples} audio samples in this chunk")
            
            offset = 0
            parsed_count = 0
            while offset + record_size <= len(payload):
                # Read timestamp
                timestamp = struct.unpack('<d', payload[offset:offset+8])[0]
                offset += 8
                
                # Read channel data
                sample_data = []
                for _ in range(channel_count):
                    value = struct.unpack('<f', payload[offset:offset+4])[0]
                    sample_data.append(value)
                    offset += 4
                    
                samples.append((timestamp, sample_data))
                parsed_count += 1
                
            print(f"  ✅ Parsed {parsed_count} audio samples successfully")
                
        except struct.error as e:
            print(f"  ⚠️ Error parsing audio samples: {e}")
            
        return samples
    
    def parse_mouse_samples(self, payload: bytes, channel_count: int) -> List[Tuple[float, List[float]]]:
        """Parse mouse sample data from payload."""
        print(f"  🖱️  Parsing mouse samples (payload size: {len(payload)} bytes, {channel_count} channels)...")
        samples = []
        try:
            sample_size = 4  # float32
            timestamp_size = 8
            record_size = timestamp_size + (channel_count * sample_size)
            expected_samples = len(payload) // record_size
            print(f"    Expected {expected_samples} mouse events in this chunk")
            
            offset = 0
            parsed_count = 0
            while offset + record_size <= len(payload):
                # Read timestamp
                timestamp = struct.unpack('<d', payload[offset:offset+8])[0]
                offset += 8
                
                # Read channel data (rel_time, x, y)
                sample_data = []
                for _ in range(channel_count):
                    value = struct.unpack('<f', payload[offset:offset+4])[0]
                    sample_data.append(value)
                    offset += 4
                    
                samples.append((timestamp, sample_data))
                parsed_count += 1
                
            print(f"  ✅ Parsed {parsed_count} mouse events successfully")
            if parsed_count > 0:
                first_event = samples[0]
                print(f"    First event: timestamp={first_event[0]:.6f}, x={first_event[1][1]:.1f}, y={first_event[1][2]:.1f}")
                
        except struct.error as e:
            print(f"  ⚠️ Error parsing mouse samples: {e}")
            
        return samples
    
    def detect_calibration_in_chunk(self, chunk_samples: List[Tuple[float, List[float]]]) -> Optional[float]:
        """
        Analyze a chunk of audio samples to detect calibration beep pattern.
        Returns calibration end time if detected, None otherwise.
        """
        print(f"  🔍 Starting calibration detection on chunk...")
        
        if not chunk_samples:
            print("  ❌ No samples in chunk, skipping detection")
            return None
            
        # Convert to arrays for analysis
        timestamps = [s[0] for s in chunk_samples]
        chunk_start_time = timestamps[0]
        chunk_end_time = timestamps[-1]
        chunk_duration = chunk_end_time - chunk_start_time
        
        print(f"    Chunk timespan: {chunk_start_time:.6f} to {chunk_end_time:.6f} ({chunk_duration:.3f}s)")
        
        if HAS_NUMPY:
            print("    Using NumPy-based FFT analysis...")
            audio_data = np.array([s[1] for s in chunk_samples], dtype=np.float32)
            n_samples, n_channels = audio_data.shape
            print(f"    Audio data shape: {n_samples} samples × {n_channels} channels")
        else:
            print("    Using pure Python analysis...")
            audio_data = [s[1] for s in chunk_samples]
            n_samples = len(audio_data)
            n_channels = len(audio_data[0]) if n_samples > 0 else 1
            print(f"    Audio data: {n_samples} samples × {n_channels} channels")
        
        # Expected calibration parameters
        expected_duration = 0.5  # 500ms total calibration
        burst_duration = 0.02    # 20ms tone
        
        print(f"    Looking for: {burst_duration*1000:.0f}ms bursts in {expected_duration*1000:.0f}ms total sequence")
        
        if HAS_NUMPY and n_samples > 0:
            # Use FFT to detect 1 kHz content
            window_size = min(n_samples, int(burst_duration * self.sample_rate))
            if window_size < 100:  # Too small for reliable FFT
                print(f"    ⚠️ Window size too small ({window_size} samples) for reliable FFT analysis")
                return None
                
            print(f"    FFT window size: {window_size} samples ({window_size/self.sample_rate*1000:.1f}ms)")
            
            # Analyze overlapping windows
            step_size = window_size // 4
            tone_detections = []
            energy_levels = []
            
            print(f"    Scanning with {step_size} sample steps...")
            
            for start_idx in range(0, n_samples - window_size, step_size):
                # Get window (use left channel)
                window = audio_data[start_idx:start_idx + window_size, 0]
                
                # FFT analysis
                freqs = np.fft.fftfreq(window_size, 1/self.sample_rate)
                fft_mag = np.abs(np.fft.fft(window))
                
                # Find 1 kHz bin
                target_bin = np.argmin(np.abs(freqs - 1000))
                tone_strength = fft_mag[target_bin]
                
                # RMS energy
                rms_energy = np.sqrt(np.mean(window**2))
                energy_levels.append(rms_energy)
                
                # Detection criteria
                if tone_strength > 50 and rms_energy > 0.005:  # Adjust thresholds as needed
                    window_time = timestamps[start_idx]
                    tone_detections.append(window_time)
                    print(f"      ✓ Tone detected at {window_time:.6f}s (strength: {tone_strength:.1f}, energy: {rms_energy:.6f})")
            
            avg_energy = np.mean(energy_levels)
            max_energy = np.max(energy_levels)
            print(f"    Energy analysis: avg={avg_energy:.6f}, max={max_energy:.6f}")
            
            # Need at least 3 tone detections for a valid calibration sequence
            if len(tone_detections) >= 3:
                print(f"  ✅ Found {len(tone_detections)} tone detections - calibration sequence likely present!")
                
                # Look for where stereo channels diverge (end of calibration)
                if n_channels >= 2:
                    print(f"    Searching for channel divergence (stereo → different content)...")
                    
                    for i in range(window_size, n_samples - window_size, self.sample_rate // 100):
                        if i + window_size < n_samples:
                            left_window = audio_data[i:i + window_size, 0]
                            right_window = audio_data[i:i + window_size, 1]
                            
                            # Calculate correlation
                            correlation = np.corrcoef(left_window, right_window)[0, 1]
                            
                            if correlation < 0.7:  # Channels diverged
                                calib_end_time = timestamps[i]  # Absolute timestamp
                                relative_end = calib_end_time - chunk_start_time
                                print(f"      ✅ Channel divergence found at {relative_end:.3f}s into chunk")
                                print(f"      Correlation dropped to {correlation:.3f} (threshold: 0.7)")
                                print(f"  🎯 CALIBRATION END DETECTED: {calib_end_time:.6f}")
                                return calib_end_time
                    
                    print(f"    No clear channel divergence found, using fallback method...")
                
                # Fallback: use expected duration from first detection
                calib_end_absolute = tone_detections[0] + expected_duration
                print(f"    Using expected duration fallback: first_tone + {expected_duration:.3f}s")
                print(f"  🎯 CALIBRATION END ESTIMATED: {calib_end_absolute:.6f}")
                return calib_end_absolute
            else:
                print(f"  ❌ Only {len(tone_detections)} tone detections found (need ≥3 for calibration)")
        
        else:  # Pure Python fallback
            print("    Using simple energy-based detection...")
            # Simple energy-based detection
            high_energy_count = 0
            total_energy = 0
            
            for sample_data in audio_data:
                energy = sum(abs(x) for x in sample_data) / len(sample_data)
                total_energy += energy
                if energy > 0.02:  # Threshold
                    high_energy_count += 1
            
            avg_energy = total_energy / len(audio_data)
            high_energy_ratio = high_energy_count / n_samples
            
            print(f"    Energy analysis: avg={avg_energy:.6f}, high_energy_ratio={high_energy_ratio:.3f}")
            
            # If significant portion has high energy, assume calibration present
            if high_energy_count > n_samples * 0.1:  # 10% of samples
                calib_end_time = timestamps[0] + expected_duration
                print(f"  ✅ Energy-based detection: calibration likely present")
                print(f"  🎯 CALIBRATION END ESTIMATED: {calib_end_time:.6f}")
                return calib_end_time
            else:
                print(f"  ❌ Insufficient high-energy samples for calibration detection")
        
        print("  ❌ No calibration detected in this chunk")
        return None
    
    def process_streaming(self) -> bool:
        """
        Process XDF file chunk by chunk until calibration is detected.
        Returns True if calibration was found, False otherwise.
        """
        print(f"🔍 Starting streaming XDF analysis...")
        print(f"📊 Target: Find 5-burst 1kHz calibration sequence and stop")
        
        chunk_count = 0
        audio_chunks_processed = 0
        mouse_chunks_processed = 0
        bytes_read = 0
        
        print(f"📖 Beginning chunk-by-chunk file reading...")
        
        while True:
            try:
                # Read chunk header
                print(f"\n📦 Reading chunk #{chunk_count + 1}...")
                length_bytes = self.file_handle.read(4)
                if len(length_bytes) < 4:
                    print("📄 Reached end of file")
                    break
                    
                length = struct.unpack('<I', length_bytes)[0]
                bytes_read += 4
                
                if length == 0:
                    print("  ⏭️  Empty chunk, skipping...")
                    continue
                    
                tag = struct.unpack('<H', self.file_handle.read(2))[0]
                payload = self.file_handle.read(length - 2)
                bytes_read += 2 + (length - 2)
                chunk_count += 1
                
                print(f"  📋 Chunk info: length={length}, tag={tag}, payload={len(payload)} bytes")
                print(f"  📊 Total bytes read so far: {bytes_read:,}")
                
                if tag == 2:  # Stream header
                    print("  🏷️  Processing STREAM HEADER...")
                    stream_info = self.parse_stream_header(payload)
                    stream_id = len(self.streams_info)
                    self.streams_info[stream_id] = stream_info
                    
                    # Identify stream types
                    stream_type = stream_info.get('type', [''])[0]
                    stream_name = stream_info.get('name', [''])[0]
                    
                    if stream_type == 'Audio':
                        self.audio_stream_id = stream_id
                        self.sample_rate = int(float(stream_info.get('nominal_srate', ['44100'])[0]))
                        channel_count = int(stream_info.get('channel_count', ['2'])[0])
                        print(f"📻 🎉 AUDIO STREAM IDENTIFIED!")
                        print(f"     Name: {stream_name}")
                        print(f"     Sample rate: {self.sample_rate} Hz")
                        print(f"     Channels: {channel_count}")
                        print(f"     Stream ID: {stream_id}")
                        
                    elif stream_type == 'MouseEvents':
                        self.mouse_stream_id = stream_id
                        channel_count = int(stream_info.get('channel_count', ['3'])[0])
                        print(f"🖱️  🎉 MOUSE STREAM IDENTIFIED!")
                        print(f"     Name: {stream_name}")
                        print(f"     Channels: {channel_count}")
                        print(f"     Stream ID: {stream_id}")
                    
                    else:
                        print(f"  ❓ Unknown stream type: {stream_type} ({stream_name})")
                
                elif tag == 3:  # Samples
                    print("  📊 Processing SAMPLE DATA...")
                    
                    # Determine which stream this belongs to by checking recent headers
                    # (This is a simplification - in real XDF, stream ID is embedded)
                    
                    if self.audio_stream_id is not None and not self.calibration_detected:
                        print("  🎵 Attempting to process as AUDIO samples...")
                        # Process audio samples to look for calibration
                        audio_info = self.streams_info[self.audio_stream_id]
                        channel_count = int(audio_info.get('channel_count', ['2'])[0])
                        
                        chunk_samples = self.parse_audio_samples(payload, channel_count)
                        if chunk_samples:
                            audio_chunks_processed += 1
                            print(f"  🎵 Audio chunk #{audio_chunks_processed} processed successfully")
                            
                            # Set audio start timestamp from first chunk
                            if self.audio_start_timestamp is None:
                                self.audio_start_timestamp = chunk_samples[0][0]
                                print(f"📅 🎯 AUDIO START TIMESTAMP SET: {self.audio_start_timestamp:.6f}")
                                print(f"     This will be our reference point for calibration timing")
                            
                            # Check for calibration in this chunk
                            print(f"🔍 🎯 RUNNING CALIBRATION DETECTION ON CHUNK #{audio_chunks_processed}...")
                            calib_end = self.detect_calibration_in_chunk(chunk_samples)
                            
                            if calib_end is not None:
                                self.calibration_end_time = calib_end
                                self.calibration_detected = True
                                calib_duration = calib_end - self.audio_start_timestamp
                                print(f"🎯 🎉 🎉 CALIBRATION SEQUENCE DETECTED! 🎉 🎉")
                                print(f"     Calibration end time: {calib_end:.6f}")
                                print(f"     Calibration duration: {calib_duration:.3f}s")
                                print(f"     Detection efficiency: Only {audio_chunks_processed} audio chunks needed!")
                                print(f"⚡ ⚡ STOPPING AUDIO PROCESSING - CALIBRATION FOUND! ⚡ ⚡")
                                break  # Stop processing audio chunks
                            else:
                                print(f"  ❌ No calibration detected in chunk #{audio_chunks_processed}")
                        else:
                            print(f"  ⚠️ Failed to parse audio samples from chunk")
                    
                    # Always try to collect mouse data (it's small)
                    if self.mouse_stream_id is not None:
                        print("  🖱️  Attempting to process as MOUSE samples...")
                        mouse_info = self.streams_info[self.mouse_stream_id]
                        channel_count = int(mouse_info.get('channel_count', ['3'])[0])
                        
                        mouse_samples = self.parse_mouse_samples(payload, channel_count)
                        if mouse_samples:
                            mouse_chunks_processed += 1
                            before_count = len(self.all_mouse_data)
                            self.all_mouse_data.extend(mouse_samples)
                            after_count = len(self.all_mouse_data)
                            print(f"  🖱️  Mouse chunk #{mouse_chunks_processed}: added {after_count - before_count} events (total: {after_count})")
                
                elif tag == 6:  # Stream footer
                    print("  🏁 Stream footer encountered")
                    continue
                    
                else:
                    print(f"  ❓ Unknown chunk tag: {tag}")
                
                # Stop if we've processed enough chunks without finding calibration
                if audio_chunks_processed > 50 and not self.calibration_detected:  # ~5-10 seconds
                    print(f"⚠️  🛑 TIMEOUT: No calibration found after {audio_chunks_processed} audio chunks")
                    print(f"     Using default calibration duration (500ms)")
                    if self.audio_start_timestamp:
                        self.calibration_end_time = self.audio_start_timestamp + 0.5  # 500ms default
                        self.calibration_detected = True
                        print(f"🎯 DEFAULT CALIBRATION SET: {self.calibration_end_time:.6f}")
                    break
                    
            except struct.error as e:
                print(f"❌ Struct parsing error: {e}")
                break
            except Exception as e:
                print(f"❌ Unexpected error: {e}")
                break
        
        # Continue reading the rest of the file for mouse data only
        if self.calibration_detected and self.mouse_stream_id is not None:
            print(f"\n🔄 PHASE 2: Collecting remaining mouse data...")
            print(f"   (Calibration found - skipping all audio, only reading mouse events)")
            mouse_chunks_collected = 0
            phase2_bytes = 0
            
            while True:
                try:
                    length_bytes = self.file_handle.read(4)
                    if len(length_bytes) < 4:
                        print("📄 Reached end of file during mouse collection")
                        break
                    length = struct.unpack('<I', length_bytes)[0]
                    phase2_bytes += 4
                    
                    if length == 0:
                        continue
                        
                    tag = struct.unpack('<H', self.file_handle.read(2))[0]
                    payload = self.file_handle.read(length - 2)
                    phase2_bytes += 2 + (length - 2)
                    
                    if tag == 3:  # Samples - only collect mouse data now
                        mouse_info = self.streams_info[self.mouse_stream_id]
                        channel_count = int(mouse_info.get('channel_count', ['3'])[0])
                        
                        mouse_samples = self.parse_mouse_samples(payload, channel_count)
                        if mouse_samples:
                            before_count = len(self.all_mouse_data)
                            self.all_mouse_data.extend(mouse_samples)
                            after_count = len(self.all_mouse_data)
                            mouse_chunks_collected += 1
                            
                            if mouse_chunks_collected % 10 == 0:  # Print every 10th chunk
                                print(f"  📊 Mouse collection progress: {mouse_chunks_collected} chunks, {after_count} total events")
                        
                        # Skip audio samples completely now - they would be parsed but ignored
                        
                except struct.error:
                    print("📄 Reached end of file during phase 2")
                    break
                    
            print(f"✅ Phase 2 complete: {mouse_chunks_collected} additional mouse chunks, {phase2_bytes:,} bytes")
        
        print(f"\n📈 🎯 FINAL PROCESSING STATISTICS:")
        print(f"   📦 Total chunks processed: {chunk_count}")
        print(f"   🎵 Audio chunks analyzed: {audio_chunks_processed}")
        print(f"   🖱️  Mouse chunks collected: {mouse_chunks_processed + mouse_chunks_collected}")
        print(f"   📊 Total mouse events: {len(self.all_mouse_data)}")
        print(f"   💽 Total bytes read: {bytes_read + phase2_bytes if 'phase2_bytes' in locals() else bytes_read:,}")
        print(f"   🎯 Calibration detected: {'✅ YES' if self.calibration_detected else '❌ NO'}")
        
        if self.calibration_detected:
            efficiency_ratio = audio_chunks_processed / max(chunk_count, 1) * 100
            print(f"   ⚡ Efficiency: Stopped after {efficiency_ratio:.1f}% of potential audio processing!")
        
        return self.calibration_detected

# ============================================================================
# MAIN PROCESSING FUNCTIONS
# ============================================================================

def process_xdf_streaming(xdf_path: str, out_dir: str) -> None:
    """
    Process XDF file using streaming approach to detect calibration efficiently.
    """
    print(f"🚀 ═══════════════════════════════════════════════════════════")
    print(f"🚀 STARTING ULTRA-EFFICIENT STREAMING XDF PROCESSING")
    print(f"🚀 ═══════════════════════════════════════════════════════════")
    
    out_dir = pathlib.Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"📁 Input file: {pathlib.Path(xdf_path).name}")
    print(f"📁 Output directory: {out_dir}")
    print(f"📊 File size: {os.path.getsize(xdf_path):,} bytes ({os.path.getsize(xdf_path)/1024/1024:.1f} MB)")
    
    print(f"\n🔧 Initializing streaming processor...")
    
    with StreamingXDFProcessor(xdf_path) as processor:
        print(f"🔄 Starting chunk-by-chunk processing...")
        # Process file chunk by chunk
        success = processor.process_streaming()
        
        if not success:
            print("❌ ═══════════════════════════════════════════════════════════")
            print("❌ PROCESSING FAILED")
            print("❌ ═══════════════════════════════════════════════════════════")
            print("❌ Failed to detect calibration or process streams")
            return
        
        print(f"\n📋 ═══════════════════════════════════════════════════════════")
        print(f"📋 EXTRACTING STREAM INFORMATION")
        print(f"📋 ═══════════════════════════════════════════════════════════")
        
        # Extract stream information
        audio_info = processor.streams_info.get(processor.audio_stream_id, {})
        mouse_info = processor.streams_info.get(processor.mouse_stream_id, {})
        
        audio_name = audio_info.get('name', ['UnknownAudio'])[0]
        mouse_name = mouse_info.get('name', ['UnknownMouse'])[0]
        
        print(f"🎵 Audio Stream Details:")
        print(f"   - Name: {audio_name}")
        print(f"   - Sample Rate: {processor.sample_rate} Hz")  
        print(f"   - Channels: {audio_info.get('channel_count', ['unknown'])[0]}")
        print(f"🖱️  Mouse Stream Details:")
        print(f"   - Name: {mouse_name}")
        print(f"   - Channels: {mouse_info.get('channel_count', ['unknown'])[0]}")
        
        # Calculate calibration timing
        calib_duration = processor.calibration_end_time - processor.audio_start_timestamp
        
        print(f"\n🎯 ═══════════════════════════════════════════════════════════")
        print(f"🎯 CALIBRATION ANALYSIS RESULTS")
        print(f"🎯 ═══════════════════════════════════════════════════════════")
        print(f"📅 Audio recording start: {processor.audio_start_timestamp:.6f}")
        print(f"🔚 Calibration sequence end: {processor.calibration_end_time:.6f}")
        print(f"⏱️  Calibration duration: {calib_duration:.3f}s")
        print(f"🎯 New timeline zero point: {processor.calibration_end_time:.6f}")
        
        print(f"\n🖱️  ═══════════════════════════════════════════════════════════")
        print(f"🖱️  CORRECTING MOUSE TIMESTAMPS")
        print(f"🖱️  ═══════════════════════════════════════════════════════════")
        
        # Correct mouse timestamps
        print(f"📊 Processing {len(processor.all_mouse_data)} mouse events...")
        corrected_mouse_data = []
        
        for i, (timestamp, mouse_pos) in enumerate(processor.all_mouse_data):
            corrected_time = timestamp - processor.calibration_end_time
            phase = 'calibration' if corrected_time < 0 else ('early' if corrected_time < 5 else 'experiment')
            
            corrected_mouse_data.append({
                'original_timestamp': timestamp,
                'corrected_timestamp': corrected_time,
                'x_px': mouse_pos[1] if len(mouse_pos) > 1 else 0,
                'y_px': mouse_pos[2] if len(mouse_pos) > 2 else 0,
                'phase': phase
            })
            
            # Show progress for large datasets
            if len(processor.all_mouse_data) > 100 and (i + 1) % 50 == 0:
                print(f"  📈 Progress: {i + 1}/{len(processor.all_mouse_data)} events processed...")
        
        # Analyze corrected data
        calib_clicks = sum(1 for entry in corrected_mouse_data if entry['phase'] == 'calibration')
        early_clicks = sum(1 for entry in corrected_mouse_data if entry['phase'] == 'early')
        exp_clicks = sum(1 for entry in corrected_mouse_data if entry['phase'] == 'experiment')
        
        print(f"✅ Timestamp correction complete!")
        print(f"📊 Click Distribution Analysis:")
        print(f"   - During calibration (t < 0): {calib_clicks}")
        print(f"   - First 5s after calibration (0 ≤ t < 5): {early_clicks}")
        print(f"   - Rest of experiment (t ≥ 5): {exp_clicks}")
        
        print(f"\n💾 ═══════════════════════════════════════════════════════════")
        print(f"💾 SAVING OUTPUT FILES")
        print(f"💾 ═══════════════════════════════════════════════════════════")
        
        # Save corrected timestamps
        timestamps_file = out_dir / f"{audio_name}_streaming_corrected_timestamps.csv"
        print(f"📝 Creating timestamps CSV: {timestamps_file.name}")
        
        with open(timestamps_file, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['original_timestamp', 'corrected_timestamp', 'x_px', 'y_px', 'phase'])
            
            for entry in corrected_mouse_data:
                writer.writerow([
                    entry['original_timestamp'],
                    entry['corrected_timestamp'], 
                    entry['x_px'],
                    entry['y_px'],
                    entry['phase']
                ])
        
        print(f"✅ Timestamps CSV saved: {len(corrected_mouse_data)} rows written")
        
        # Create summary
        summary_file = out_dir / f"{audio_name}_streaming_summary.txt"
        print(f"📄 Creating summary report: {summary_file.name}")
        
        with open(summary_file, 'w') as f:
            f.write(f"Ultra-Efficient Streaming XDF Analysis Report\n")
            f.write(f"============================================\n\n")
            f.write(f"PROCESSING DETAILS:\n")
            f.write(f"- Input File: {pathlib.Path(xdf_path).name}\n")
            f.write(f"- File Size: {os.path.getsize(xdf_path):,} bytes ({os.path.getsize(xdf_path)/1024/1024:.1f} MB)\n")
            f.write(f"- Processing Method: Chunk-by-chunk streaming\n")
            f.write(f"- Processing Date: {pathlib.Path().cwd()}\n\n")  # Could add actual date
            f.write(f"STREAM INFORMATION:\n")
            f.write(f"- Audio Stream: {audio_name}\n")
            f.write(f"- Mouse Stream: {mouse_name}\n")
            f.write(f"- Sample Rate: {processor.sample_rate} Hz\n\n")
            f.write(f"EFFICIENCY METRICS:\n")
            f.write(f"- Streaming approach used: YES\n")
            f.write(f"- Stopped after calibration detection: YES\n")
            f.write(f"- Full audio decode avoided: YES\n")
            f.write(f"- Early termination achieved: YES\n\n")
            f.write(f"CALIBRATION DETECTION:\n")
            f.write(f"- Calibration duration: {calib_duration:.3f}s\n")
            f.write(f"- Audio start timestamp: {processor.audio_start_timestamp:.6f}\n")
            f.write(f"- Calibration end timestamp: {processor.calibration_end_time:.6f}\n")
            f.write(f"- New timeline zero point: {processor.calibration_end_time:.6f}\n\n")
            f.write(f"MOUSE CLICK ANALYSIS:\n")
            f.write(f"- Total mouse events: {len(corrected_mouse_data)}\n")
            f.write(f"- During calibration (t < 0): {calib_clicks}\n")
            f.write(f"- First 5s after calibration (0 ≤ t < 5): {early_clicks}\n")
            f.write(f"- Rest of experiment (t ≥ 5): {exp_clicks}\n\n")
            f.write(f"TIMESTAMP CORRECTION:\n")
            f.write(f"- All timestamps adjusted relative to calibration end\n")
            f.write(f"- New t=0 corresponds to original timestamp: {processor.calibration_end_time:.6f}\n")
            f.write(f"- Calibration beep duration removed: {calib_duration:.3f}s\n")
        
        print(f"✅ Summary report saved")
        
        # Print final results
        print(f"\n🎉 ═══════════════════════════════════════════════════════════")
        print(f"🎉 STREAMING ANALYSIS SUCCESSFULLY COMPLETED!")
        print(f"🎉 ═══════════════════════════════════════════════════════════")
        
        print(f"📊 Final Results Summary:")
        print(f"   🎯 Calibration duration: {calib_duration:.3f}s")
        print(f"   🖱️  Mouse clicks corrected: {len(corrected_mouse_data)}")
        print(f"   📈 Calibration phase clicks: {calib_clicks}")
        print(f"   📈 Early experiment clicks: {early_clicks}")
        print(f"   📈 Total experiment clicks: {exp_clicks}")
        
        print(f"\n📁 Output Files Created:")
        print(f"   📝 Corrected timestamps: {timestamps_file.name}")
        print(f"   📄 Analysis summary: {summary_file.name}")
        
        print(f"\n⚡ Efficiency Achievements:")
        print(f"   ✅ Stopped processing audio immediately after calibration detection")
        print(f"   ✅ Avoided decoding the entire audio stream")
        print(f"   ✅ Processed only the minimum necessary data")
        print(f"   🎯 New experiment timeline: t=0 at original {processor.calibration_end_time:.6f}")
        print(f"   💾 Saved {calib_duration:.3f}s of calibration beep time")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print(f"🎬 ═══════════════════════════════════════════════════════════")
    print(f"🎬 XDF CALIBRATION DETECTOR - STREAMING VERSION")
    print(f"🎬 Ultra-Efficient Chunk-by-Chunk Processing")
    print(f"🎬 ═══════════════════════════════════════════════════════════")
    
    # ---- USER INPUTS -------------------------------------------------------
    xdf_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf"
    out_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT"
    # -----------------------------------------------------------------------
    
    print(f"🔧 Configuration:")
    print(f"   📁 Input XDF: {pathlib.Path(xdf_path).name}")
    print(f"   📁 Output Dir: {pathlib.Path(out_dir).name}")
    
    # Check if file exists
    print(f"\n🔍 Validating input file...")
    if not os.path.exists(xdf_path):
        print(f"❌ ═══════════════════════════════════════════════════════════")
        print(f"❌ ERROR: FILE NOT FOUND")
        print(f"❌ ═══════════════════════════════════════════════════════════")
        print(f"❌ XDF file not found: {xdf_path}")
        print(f"💡 Please update the xdf_path variable in the script.")
        exit(1)
    
    file_size_mb = os.path.getsize(xdf_path) / 1024 / 1024
    print(f"✅ Input file validated: {file_size_mb:.1f} MB")
    
    print(f"\n🔍 Validating output directory...")
    try:
        pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
        print(f"✅ Output directory ready: {out_dir}")
    except Exception as e:
        print(f"❌ Failed to create output directory: {e}")
        exit(1)
    
    print(f"\n🚀 Launching streaming processor...")
    
    # Process using ultra-efficient streaming approach
    try:
        process_xdf_streaming(xdf_path, out_dir)
    except Exception as e:
        print(f"\n❌ ═══════════════════════════════════════════════════════════")
        print(f"❌ PROCESSING ERROR")
        print(f"❌ ═══════════════════════════════════════════════════════════")
        print(f"❌ An error occurred during processing: {e}")
        print(f"💡 Check the debug output above for details.")
        exit(1)
    
    print(f"\n🎉 ═══════════════════════════════════════════════════════════")
    print(f"🎉 ULTRA-EFFICIENT PROCESSING COMPLETE!")
    print(f"🎉 ═══════════════════════════════════════════════════════════")
    print(f"🎯 Mission accomplished: Calibration detected and timestamps corrected!")
    print(f"⚡ Maximum efficiency achieved through streaming approach!")
    print(f"📁 Check your output directory for results!")
    print(f"🎉 ═══════════════════════════════════════════════════════════")

🎬 ═══════════════════════════════════════════════════════════
🎬 XDF CALIBRATION DETECTOR - STREAMING VERSION
🎬 Ultra-Efficient Chunk-by-Chunk Processing
🎬 ═══════════════════════════════════════════════════════════
🔧 Configuration:
   📁 Input XDF: P001_20250506_171104_R001.xdf
   📁 Output Dir: PPS_RT

🔍 Validating input file...
✅ Input file validated: 964.6 MB

🔍 Validating output directory...
✅ Output directory ready: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT

🚀 Launching streaming processor...
🚀 ═══════════════════════════════════════════════════════════
🚀 STARTING ULTRA-EFFICIENT STREAMING XDF PROCESSING
🚀 ═══════════════════════════════════════════════════════════
📁 Input file: P001_20250506_171104_R001.xdf
📁 Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT
📊 File size: 1,011,456,099 bytes (964.6 MB)

🔧 Initializing streaming processor...
🔧 Opening XDF file...
✅ File opened successfully: 

In [5]:
#!/usr/bin/env python3
"""
Ultra-Efficient XDF Calibration Detector

Core principle: Extract ONLY first 30 seconds of audio to detect calibration,
then skip ALL remaining audio. Process mouse data separately.

Efficiency targets:
- Memory usage: <50MB peak
- Processing time: <60 seconds for 1GB files  
- Audio processed: ~0.1% of total (30s out of hours)
- Disk usage: 0 bytes (no temp files)
"""

import os
import struct
import csv
from typing import Optional, Tuple, List, Dict, BinaryIO
import xml.etree.ElementTree as ET

try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False
    print("Warning: NumPy not available. Using fallback detection.")

# Configuration
MAX_AUDIO_SECONDS = 30  # Only process first 30 seconds
SAMPLE_RATE = 44100  # Expected sample rate
CALIBRATION_DURATION = 0.5  # Expected calibration duration
DEBUG = False  # Set True for verbose debugging

def debug_print(msg: str):
    """Print only if debugging enabled"""
    if DEBUG:
        print(msg)

class UltraEfficientXDFProcessor:
    """Minimal memory footprint XDF processor"""
    
    def __init__(self, filepath: str):
        self.filepath = filepath
        self.file: Optional[BinaryIO] = None
        self.streams: Dict[int, dict] = {}
        self.audio_id: Optional[int] = None
        self.mouse_id: Optional[int] = None
        self.audio_start_time: Optional[float] = None
        self.calibration_end_time: Optional[float] = None
        self.mouse_events: List[Tuple[float, float, float]] = []
        self.audio_buffer = []  # Temporary buffer for audio samples
        self.audio_seconds_collected = 0.0
        
    def __enter__(self):
        self.file = open(self.filepath, 'rb')
        # Verify XDF signature
        if self.file.read(4) != b'XDF:':
            raise ValueError("Invalid XDF file")
        return self
        
    def __exit__(self, *args):
        if self.file:
            self.file.close()
            
    def read_chunk_header(self) -> Optional[Tuple[int, int]]:
        """Read chunk length and tag"""
        length_bytes = self.file.read(4)
        if len(length_bytes) < 4:
            return None
        length = struct.unpack('<I', length_bytes)[0]
        if length == 0:
            return 0, 0
        tag = struct.unpack('<H', self.file.read(2))[0]
        return length - 2, tag
        
    def parse_header_minimal(self, data: bytes, stream_id: int) -> dict:
        """Extract only essential info from stream header"""
        try:
            xml = ET.fromstring(data.decode('utf-8'))
            info = {
                'id': stream_id,
                'type': xml.findtext('.//type', ''),
                'name': xml.findtext('.//name', ''),
                'channel_count': int(xml.findtext('.//channel_count', '1')),
                'nominal_srate': float(xml.findtext('.//nominal_srate', '44100'))
            }
            return info
        except:
            return {'id': stream_id, 'type': '', 'name': '', 'channel_count': 1}
            
    def detect_calibration_numpy(self, samples: np.ndarray, timestamps: np.ndarray) -> Optional[float]:
        """Fast NumPy-based calibration detection"""
        if len(samples) < SAMPLE_RATE * 0.1:  # Need at least 100ms
            return None
            
        # Simple but effective: look for high energy bursts
        window = int(0.02 * SAMPLE_RATE)  # 20ms windows
        step = window // 4
        
        # Calculate windowed RMS energy
        high_energy_windows = []
        for i in range(0, len(samples) - window, step):
            chunk = samples[i:i+window, 0]  # Use first channel
            rms = np.sqrt(np.mean(chunk**2))
            if rms > 0.01:  # Energy threshold
                high_energy_windows.append(i)
                
        # If we found 3+ high energy windows in first second, likely calibration
        if len(high_energy_windows) >= 3:
            # Calibration end is ~500ms after first detection
            first_detection_idx = high_energy_windows[0]
            calib_end_idx = min(first_detection_idx + int(CALIBRATION_DURATION * SAMPLE_RATE), 
                               len(timestamps) - 1)
            return timestamps[calib_end_idx]
            
        return None
        
    def detect_calibration_simple(self, samples: List[List[float]], timestamps: List[float]) -> Optional[float]:
        """Simple fallback calibration detection"""
        if not samples or len(samples) < 1000:
            return None
            
        # Count high-amplitude samples
        high_count = sum(1 for s in samples[:int(SAMPLE_RATE * 0.5)] 
                        if abs(s[0]) > 0.01)
        
        # If >10% samples are high amplitude in first 500ms, assume calibration
        if high_count > len(samples[:int(SAMPLE_RATE * 0.5)]) * 0.1:
            # Return timestamp 500ms after start
            return timestamps[0] + CALIBRATION_DURATION
            
        return None
        
    def process_efficient(self) -> bool:
        """Ultra-efficient processing: minimal audio, all mouse data"""
        print("🚀 Starting ultra-efficient XDF processing...")
        
        audio_found = False
        calibration_detected = False
        chunks_read = 0
        
        # PHASE 1: Find streams and collect first 30s of audio
        print("📊 Phase 1: Stream identification and minimal audio extraction")
        
        while True:
            header = self.read_chunk_header()
            if not header:
                break
                
            length, tag = header
            if length == 0:
                continue
                
            chunks_read += 1
            
            if tag == 2:  # Stream header
                stream_id = struct.unpack('<I', self.file.read(4))[0]
                info = self.parse_header_minimal(self.file.read(length - 4), stream_id)
                self.streams[stream_id] = info
                
                if info['type'] == 'Audio':
                    self.audio_id = stream_id
                    print(f"✓ Audio stream found: {info['name']} (ID: {stream_id})")
                elif info['type'] == 'MouseEvents':
                    self.mouse_id = stream_id
                    print(f"✓ Mouse stream found: {info['name']} (ID: {stream_id})")
                    
            elif tag == 3:  # Samples
                # Read stream ID
                stream_id = struct.unpack('<I', self.file.read(4))[0]
                data = self.file.read(length - 4)
                
                # Process audio samples if we haven't collected enough yet
                if stream_id == self.audio_id and self.audio_seconds_collected < MAX_AUDIO_SECONDS:
                    audio_info = self.streams[self.audio_id]
                    channels = audio_info['channel_count']
                    sample_rate = audio_info['nominal_srate']
                    
                    # Parse samples efficiently
                    record_size = 8 + (channels * 4)  # timestamp + channels * float32
                    n_samples = len(data) // record_size
                    
                    for i in range(n_samples):
                        offset = i * record_size
                        timestamp = struct.unpack('<d', data[offset:offset+8])[0]
                        
                        if self.audio_start_time is None:
                            self.audio_start_time = timestamp
                            audio_found = True
                            
                        # Only collect first 30 seconds
                        if timestamp - self.audio_start_time > MAX_AUDIO_SECONDS:
                            break
                            
                        # Extract sample values
                        values = []
                        for ch in range(channels):
                            val_offset = offset + 8 + (ch * 4)
                            val = struct.unpack('<f', data[val_offset:val_offset+4])[0]
                            values.append(val)
                            
                        self.audio_buffer.append((timestamp, values))
                        
                    self.audio_seconds_collected = (self.audio_buffer[-1][0] - 
                                                   self.audio_start_time if self.audio_buffer else 0)
                    
                    debug_print(f"  Audio collected: {self.audio_seconds_collected:.1f}s")
                    
                    # Check if we have enough audio to detect calibration
                    if self.audio_seconds_collected >= 1.0 and not calibration_detected:
                        print(f"🔍 Detecting calibration in {len(self.audio_buffer)} samples...")
                        
                        timestamps = [s[0] for s in self.audio_buffer]
                        
                        if HAS_NUMPY:
                            samples = np.array([s[1] for s in self.audio_buffer], dtype=np.float32)
                            self.calibration_end_time = self.detect_calibration_numpy(
                                samples, np.array(timestamps))
                        else:
                            samples = [s[1] for s in self.audio_buffer]
                            self.calibration_end_time = self.detect_calibration_simple(
                                samples, timestamps)
                            
                        if self.calibration_end_time:
                            calibration_detected = True
                            calib_duration = self.calibration_end_time - self.audio_start_time
                            print(f"✅ Calibration detected! Duration: {calib_duration:.3f}s")
                            print(f"⚡ Stopping audio processing - goal achieved!")
                            # Clear audio buffer to free memory
                            self.audio_buffer = []
                            
                elif stream_id == self.mouse_id:
                    # Always collect mouse data (it's tiny)
                    mouse_info = self.streams[self.mouse_id]
                    channels = mouse_info['channel_count']
                    
                    record_size = 8 + (channels * 4)
                    n_samples = len(data) // record_size
                    
                    for i in range(n_samples):
                        offset = i * record_size
                        timestamp = struct.unpack('<d', data[offset:offset+8])[0]
                        
                        # Extract x, y coordinates (channels 1 and 2)
                        x = struct.unpack('<f', data[offset+12:offset+16])[0] if channels > 1 else 0
                        y = struct.unpack('<f', data[offset+16:offset+20])[0] if channels > 2 else 0
                        
                        self.mouse_events.append((timestamp, x, y))
                        
            else:
                # Skip other chunk types quickly
                self.file.read(length)
                
            # Stop looking for audio after we have enough or found calibration
            if (calibration_detected or self.audio_seconds_collected >= MAX_AUDIO_SECONDS) and audio_found:
                break
                
        # If no calibration detected, use default
        if not calibration_detected and self.audio_start_time:
            self.calibration_end_time = self.audio_start_time + CALIBRATION_DURATION
            print(f"⚠️ Using default calibration duration: {CALIBRATION_DURATION}s")
            
        # PHASE 2: Skip remaining audio, collect only mouse data
        if calibration_detected or self.audio_start_time:
            print("\n📊 Phase 2: Collecting remaining mouse data (skipping audio)")
            
            mouse_chunks = 0
            while True:
                header = self.read_chunk_header()
                if not header:
                    break
                    
                length, tag = header
                if length == 0:
                    continue
                    
                if tag == 3:  # Samples
                    stream_id = struct.unpack('<I', self.file.read(4))[0]
                    
                    if stream_id == self.mouse_id:
                        # Process mouse data
                        data = self.file.read(length - 4)
                        mouse_info = self.streams[self.mouse_id]
                        channels = mouse_info['channel_count']
                        
                        record_size = 8 + (channels * 4)
                        n_samples = len(data) // record_size
                        
                        for i in range(n_samples):
                            offset = i * record_size
                            timestamp = struct.unpack('<d', data[offset:offset+8])[0]
                            x = struct.unpack('<f', data[offset+12:offset+16])[0] if channels > 1 else 0
                            y = struct.unpack('<f', data[offset+16:offset+20])[0] if channels > 2 else 0
                            self.mouse_events.append((timestamp, x, y))
                            
                        mouse_chunks += 1
                        if mouse_chunks % 10 == 0:
                            debug_print(f"  Mouse chunks processed: {mouse_chunks}")
                    else:
                        # Skip audio chunks entirely - just seek past them
                        self.file.seek(length - 4, 1)
                else:
                    # Skip other chunks
                    self.file.read(length)
                    
        print(f"\n✅ Processing complete!")
        print(f"   Total chunks read: {chunks_read}")
        print(f"   Audio processed: {self.audio_seconds_collected:.1f}s (target: ≤{MAX_AUDIO_SECONDS}s)")
        print(f"   Mouse events collected: {len(self.mouse_events)}")
        
        return bool(self.calibration_end_time)

def process_xdf_ultra_efficient(xdf_path: str, output_dir: str):
    """Main processing function - ultra efficient approach"""
    import pathlib
    import time
    
    start_time = time.time()
    
    print("="*60)
    print("ULTRA-EFFICIENT XDF CALIBRATION DETECTOR")
    print("="*60)
    
    # Setup paths
    xdf_path = pathlib.Path(xdf_path)
    output_dir = pathlib.Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    file_size_mb = xdf_path.stat().st_size / 1024 / 1024
    print(f"Input: {xdf_path.name} ({file_size_mb:.1f} MB)")
    print(f"Output: {output_dir}")
    
    # Process file
    with UltraEfficientXDFProcessor(str(xdf_path)) as processor:
        success = processor.process_efficient()
        
        if not success:
            print("❌ Failed to process XDF file")
            return
            
        # Extract results
        calib_duration = processor.calibration_end_time - processor.audio_start_time
        
        print(f"\n📊 Results:")
        print(f"   Calibration end: {processor.calibration_end_time:.6f}")
        print(f"   Calibration duration: {calib_duration:.3f}s")
        print(f"   New t=0: {processor.calibration_end_time:.6f}")
        
        # Correct mouse timestamps
        corrected_data = []
        for timestamp, x, y in processor.mouse_events:
            corrected_time = timestamp - processor.calibration_end_time
            phase = 'calibration' if corrected_time < 0 else 'experiment'
            corrected_data.append({
                'original_timestamp': timestamp,
                'corrected_timestamp': corrected_time,
                'x_px': x,
                'y_px': y,
                'phase': phase
            })
            
        # Save results
        audio_name = processor.streams.get(processor.audio_id, {}).get('name', 'audio')
        output_file = output_dir / f"{audio_name}_corrected_timestamps.csv"
        
        with open(output_file, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['original_timestamp', 'corrected_timestamp', 
                                                   'x_px', 'y_px', 'phase'])
            writer.writeheader()
            writer.writerows(corrected_data)
            
        # Summary
        calib_events = sum(1 for d in corrected_data if d['phase'] == 'calibration')
        exp_events = sum(1 for d in corrected_data if d['phase'] == 'experiment')
        
        print(f"\n✅ Saved {len(corrected_data)} corrected timestamps")
        print(f"   During calibration: {calib_events}")
        print(f"   During experiment: {exp_events}")
        
        # Performance metrics
        elapsed = time.time() - start_time
        print(f"\n⚡ Performance:")
        print(f"   Processing time: {elapsed:.1f}s")
        print(f"   Speed: {file_size_mb/elapsed:.1f} MB/s")
        print(f"   Efficiency: Processed ~{processor.audio_seconds_collected/elapsed:.1f}s of audio per second")
        
        # Save summary
        summary_file = output_dir / f"{audio_name}_summary.txt"
        with open(summary_file, 'w') as f:
            f.write(f"Ultra-Efficient XDF Processing Summary\n")
            f.write(f"=====================================\n\n")
            f.write(f"File: {xdf_path.name} ({file_size_mb:.1f} MB)\n")
            f.write(f"Processing time: {elapsed:.1f}s\n")
            f.write(f"Audio processed: {processor.audio_seconds_collected:.1f}s (max: {MAX_AUDIO_SECONDS}s)\n")
            f.write(f"Calibration duration: {calib_duration:.3f}s\n")
            f.write(f"Mouse events: {len(corrected_data)}\n")
            f.write(f"  - During calibration: {calib_events}\n")
            f.write(f"  - During experiment: {exp_events}\n")
            f.write(f"\nNew timeline: t=0 at {processor.calibration_end_time:.6f}\n")

if __name__ == "__main__":
    # User configuration
    xdf_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf"
    output_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT"
    
    # Enable debugging if needed
    # DEBUG = True
    
    try:
        process_xdf_ultra_efficient(xdf_path, output_dir)
    except FileNotFoundError:
        print(f"❌ File not found: {xdf_path}")
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

ULTRA-EFFICIENT XDF CALIBRATION DETECTOR
Input: P001_20250506_171104_R001.xdf (964.6 MB)
Output: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT
🚀 Starting ultra-efficient XDF processing...
📊 Phase 1: Stream identification and minimal audio extraction

✅ Processing complete!
   Total chunks read: 4
   Audio processed: 0.0s (target: ≤30s)
   Mouse events collected: 0
❌ Failed to process XDF file


In [6]:
#!/usr/bin/env python3
"""
Ultra-Efficient XDF Calibration Detector

Core principle: Extract ONLY first 30 seconds of audio to detect calibration,
then skip ALL remaining audio. Process mouse data separately.

Efficiency targets:
- Memory usage: <50MB peak
- Processing time: <60 seconds for 1GB files  
- Audio processed: ~0.1% of total (30s out of hours)
- Disk usage: 0 bytes (no temp files)
"""

import os
import struct
import csv
from typing import Optional, Tuple, List, Dict, BinaryIO
import xml.etree.ElementTree as ET

try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False
    print("Warning: NumPy not available. Using fallback detection.")

# Configuration
MAX_AUDIO_SECONDS = 30  # Only process first 30 seconds
SAMPLE_RATE = 44100  # Expected sample rate
CALIBRATION_DURATION = 0.5  # Expected calibration duration
DEBUG = False  # Set True for verbose debugging

def debug_print(msg: str):
    """Print only if debugging enabled"""
    if DEBUG:
        print(msg)

class UltraEfficientXDFProcessor:
    """Minimal memory footprint XDF processor"""
    
    def __init__(self, filepath: str):
        self.filepath = filepath
        self.file: Optional[BinaryIO] = None
        self.streams: Dict[int, dict] = {}
        self.audio_id: Optional[int] = None
        self.mouse_id: Optional[int] = None
        self.audio_start_time: Optional[float] = None
        self.calibration_end_time: Optional[float] = None
        self.mouse_events: List[Tuple[float, float, float]] = []
        self.audio_buffer = []  # Temporary buffer for audio samples
        self.audio_seconds_collected = 0.0
        
    def __enter__(self):
        self.file = open(self.filepath, 'rb')
        # Verify XDF signature
        if self.file.read(4) != b'XDF:':
            raise ValueError("Invalid XDF file")
        return self
        
    def __exit__(self, *args):
        if self.file:
            self.file.close()
            
    def read_chunk_header(self) -> Optional[Tuple[int, int]]:
        """Read chunk length and tag"""
        length_bytes = self.file.read(4)
        if len(length_bytes) < 4:
            return None
        length = struct.unpack('<I', length_bytes)[0]
        if length == 0:
            return 0, 0
        tag = struct.unpack('<H', self.file.read(2))[0]
        return length - 2, tag
        
    def parse_header_minimal(self, data: bytes, stream_id: int) -> dict:
        """Extract only essential info from stream header"""
        try:
            xml = ET.fromstring(data.decode('utf-8'))
            info = {
                'id': stream_id,
                'type': xml.findtext('.//type', ''),
                'name': xml.findtext('.//name', ''),
                'channel_count': int(xml.findtext('.//channel_count', '1')),
                'nominal_srate': float(xml.findtext('.//nominal_srate', '44100'))
            }
            return info
        except:
            return {'id': stream_id, 'type': '', 'name': '', 'channel_count': 1}
            
    def detect_calibration_numpy(self, samples: np.ndarray, timestamps: np.ndarray) -> Optional[float]:
        """Fast NumPy-based calibration detection"""
        if len(samples) < SAMPLE_RATE * 0.1:  # Need at least 100ms
            return None
            
        # Simple but effective: look for high energy bursts
        window = int(0.02 * SAMPLE_RATE)  # 20ms windows
        step = window // 4
        
        # Calculate windowed RMS energy
        high_energy_windows = []
        for i in range(0, len(samples) - window, step):
            chunk = samples[i:i+window, 0]  # Use first channel
            rms = np.sqrt(np.mean(chunk**2))
            if rms > 0.01:  # Energy threshold
                high_energy_windows.append(i)
                
        # If we found 3+ high energy windows in first second, likely calibration
        if len(high_energy_windows) >= 3:
            # Calibration end is ~500ms after first detection
            first_detection_idx = high_energy_windows[0]
            calib_end_idx = min(first_detection_idx + int(CALIBRATION_DURATION * SAMPLE_RATE), 
                               len(timestamps) - 1)
            return timestamps[calib_end_idx]
            
        return None
        
    def detect_calibration_simple(self, samples: List[List[float]], timestamps: List[float]) -> Optional[float]:
        """Simple fallback calibration detection"""
        if not samples or len(samples) < 1000:
            return None
            
        # Count high-amplitude samples
        high_count = sum(1 for s in samples[:int(SAMPLE_RATE * 0.5)] 
                        if abs(s[0]) > 0.01)
        
        # If >10% samples are high amplitude in first 500ms, assume calibration
        if high_count > len(samples[:int(SAMPLE_RATE * 0.5)]) * 0.1:
            # Return timestamp 500ms after start
            return timestamps[0] + CALIBRATION_DURATION
            
        return None
        
    def process_efficient(self) -> bool:
        """Ultra-efficient processing: minimal audio, all mouse data"""
        print("🚀 Starting ultra-efficient XDF processing...")
        
        audio_found = False
        calibration_detected = False
        chunks_read = 0
        total_chunks = 0
        samples_chunks = 0
        
        # PHASE 1: Find streams and collect first 30s of audio
        print("📊 Phase 1: Stream identification and minimal audio extraction")
        
        while True:
            header = self.read_chunk_header()
            if not header:
                break
                
            length, tag = header
            if length == 0:
                continue
                
            total_chunks += 1
            
            if tag == 2:  # Stream header
                stream_id = struct.unpack('<I', self.file.read(4))[0]
                info = self.parse_header_minimal(self.file.read(length - 4), stream_id)
                self.streams[stream_id] = info
                
                if info['type'] == 'Audio':
                    self.audio_id = stream_id
                    print(f"✓ Audio stream found: {info['name']} (ID: {stream_id}, {info['channel_count']} channels)")
                elif info['type'] == 'MouseEvents':
                    self.mouse_id = stream_id
                    print(f"✓ Mouse stream found: {info['name']} (ID: {stream_id}, {info['channel_count']} channels)")
                else:
                    print(f"  Other stream: {info['name']} (type: {info['type']})")
                    
            elif tag == 3:  # Samples
                samples_chunks += 1
                # Read stream ID
                stream_id = struct.unpack('<I', self.file.read(4))[0]
                data = self.file.read(length - 4)
                
                # Show what we're processing
                if samples_chunks <= 5 or samples_chunks % 100 == 0:
                    print(f"  Sample chunk #{samples_chunks}: stream_id={stream_id}, size={len(data)} bytes")
                
                # Process audio samples if we haven't collected enough yet
                if stream_id == self.audio_id and self.audio_id is not None and self.audio_seconds_collected < MAX_AUDIO_SECONDS:
                    audio_info = self.streams[self.audio_id]
                    channels = audio_info['channel_count']
                    sample_rate = audio_info['nominal_srate']
                    
                    # Parse samples efficiently
                    record_size = 8 + (channels * 4)  # timestamp + channels * float32
                    n_samples = len(data) // record_size
                    
                    if n_samples > 0:
                        print(f"    Processing {n_samples} audio samples...")
                    
                    samples_processed = 0
                    for i in range(n_samples):
                        offset = i * record_size
                        timestamp = struct.unpack('<d', data[offset:offset+8])[0]
                        
                        if self.audio_start_time is None:
                            self.audio_start_time = timestamp
                            audio_found = True
                            print(f"    ⏰ Audio start time: {timestamp:.6f}")
                            
                        # Only collect first 30 seconds
                        if timestamp - self.audio_start_time > MAX_AUDIO_SECONDS:
                            print(f"    ⏹️  Reached {MAX_AUDIO_SECONDS}s limit, stopping audio collection")
                            break
                            
                        # Extract sample values
                        values = []
                        for ch in range(channels):
                            val_offset = offset + 8 + (ch * 4)
                            val = struct.unpack('<f', data[val_offset:val_offset+4])[0]
                            values.append(val)
                            
                        self.audio_buffer.append((timestamp, values))
                        samples_processed += 1
                        
                    if samples_processed > 0:
                        self.audio_seconds_collected = (self.audio_buffer[-1][0] - 
                                                       self.audio_start_time if self.audio_buffer else 0)
                        print(f"    Audio collected: {self.audio_seconds_collected:.1f}s ({len(self.audio_buffer)} total samples)")
                    
                    # Check if we have enough audio to detect calibration
                    if self.audio_seconds_collected >= 1.0 and not calibration_detected:
                        print(f"🔍 Detecting calibration in {len(self.audio_buffer)} samples...")
                        
                        timestamps = [s[0] for s in self.audio_buffer]
                        
                        if HAS_NUMPY:
                            samples = np.array([s[1] for s in self.audio_buffer], dtype=np.float32)
                            self.calibration_end_time = self.detect_calibration_numpy(
                                samples, np.array(timestamps))
                        else:
                            samples = [s[1] for s in self.audio_buffer]
                            self.calibration_end_time = self.detect_calibration_simple(
                                samples, timestamps)
                            
                        if self.calibration_end_time:
                            calibration_detected = True
                            calib_duration = self.calibration_end_time - self.audio_start_time
                            print(f"✅ Calibration detected! Duration: {calib_duration:.3f}s")
                            print(f"⚡ Stopping audio processing - goal achieved!")
                            # Clear audio buffer to free memory
                            self.audio_buffer = []
                            
                elif stream_id == self.mouse_id and self.mouse_id is not None:
                    # Always collect mouse data (it's tiny)
                    mouse_info = self.streams[self.mouse_id]
                    channels = mouse_info['channel_count']
                    
                    record_size = 8 + (channels * 4)
                    n_samples = len(data) // record_size
                    
                    if n_samples > 0 and len(self.mouse_events) == 0:
                        print(f"    Processing {n_samples} mouse events...")
                    
                    for i in range(n_samples):
                        offset = i * record_size
                        timestamp = struct.unpack('<d', data[offset:offset+8])[0]
                        
                        # Extract x, y coordinates (channels 1 and 2)
                        x = struct.unpack('<f', data[offset+12:offset+16])[0] if channels > 1 else 0
                        y = struct.unpack('<f', data[offset+16:offset+20])[0] if channels > 2 else 0
                        
                        self.mouse_events.append((timestamp, x, y))
                        
            elif tag == 6:  # Stream footer
                print(f"  Stream footer (chunk #{total_chunks})")
                self.file.read(length)
            else:
                # Skip other chunk types quickly
                print(f"  Unknown chunk type {tag} (chunk #{total_chunks})")
                self.file.read(length)
                
            chunks_read += 1
                
            # Stop looking for audio after we have enough or found calibration
            if (calibration_detected or self.audio_seconds_collected >= MAX_AUDIO_SECONDS) and audio_found:
                print(f"\n⚡ Phase 1 complete: Processed {chunks_read} chunks")
                break
                
        # If no calibration detected, use default
        if not calibration_detected and self.audio_start_time:
            self.calibration_end_time = self.audio_start_time + CALIBRATION_DURATION
            print(f"⚠️ Using default calibration duration: {CALIBRATION_DURATION}s")
            
        # PHASE 2: Skip remaining audio, collect only mouse data
        if calibration_detected or self.audio_start_time:
            print("\n📊 Phase 2: Collecting remaining mouse data (skipping audio)")
            
            mouse_chunks = 0
            while True:
                header = self.read_chunk_header()
                if not header:
                    break
                    
                length, tag = header
                if length == 0:
                    continue
                    
                if tag == 3:  # Samples
                    stream_id = struct.unpack('<I', self.file.read(4))[0]
                    
                    if stream_id == self.mouse_id:
                        # Process mouse data
                        data = self.file.read(length - 4)
                        mouse_info = self.streams[self.mouse_id]
                        channels = mouse_info['channel_count']
                        
                        record_size = 8 + (channels * 4)
                        n_samples = len(data) // record_size
                        
                        for i in range(n_samples):
                            offset = i * record_size
                            timestamp = struct.unpack('<d', data[offset:offset+8])[0]
                            x = struct.unpack('<f', data[offset+12:offset+16])[0] if channels > 1 else 0
                            y = struct.unpack('<f', data[offset+16:offset+20])[0] if channels > 2 else 0
                            self.mouse_events.append((timestamp, x, y))
                            
                        mouse_chunks += 1
                        if mouse_chunks % 10 == 0:
                            debug_print(f"  Mouse chunks processed: {mouse_chunks}")
                    else:
                        # Skip audio chunks entirely - just seek past them
                        self.file.seek(length - 4, 1)
                else:
                    # Skip other chunks
                    self.file.read(length)
                    
        print(f"\n✅ Processing complete!")
        print(f"   Total chunks read: {chunks_read}")
        print(f"   Audio processed: {self.audio_seconds_collected:.1f}s (target: ≤{MAX_AUDIO_SECONDS}s)")
        print(f"   Mouse events collected: {len(self.mouse_events)}")
        
        return bool(self.calibration_end_time)

def process_xdf_ultra_efficient(xdf_path: str, output_dir: str):
    """Main processing function - ultra efficient approach"""
    import pathlib
    import time
    
    start_time = time.time()
    
    print("="*60)
    print("ULTRA-EFFICIENT XDF CALIBRATION DETECTOR")
    print("="*60)
    
    # Setup paths
    xdf_path = pathlib.Path(xdf_path)
    output_dir = pathlib.Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    file_size_mb = xdf_path.stat().st_size / 1024 / 1024
    print(f"Input: {xdf_path.name} ({file_size_mb:.1f} MB)")
    print(f"Output: {output_dir}")
    
    # Process file
    with UltraEfficientXDFProcessor(str(xdf_path)) as processor:
        success = processor.process_efficient()
        
        if not success:
            print("❌ Failed to process XDF file")
            return
            
        # Extract results
        calib_duration = processor.calibration_end_time - processor.audio_start_time
        
        print(f"\n📊 Results:")
        print(f"   Calibration end: {processor.calibration_end_time:.6f}")
        print(f"   Calibration duration: {calib_duration:.3f}s")
        print(f"   New t=0: {processor.calibration_end_time:.6f}")
        
        # Correct mouse timestamps
        corrected_data = []
        for timestamp, x, y in processor.mouse_events:
            corrected_time = timestamp - processor.calibration_end_time
            phase = 'calibration' if corrected_time < 0 else 'experiment'
            corrected_data.append({
                'original_timestamp': timestamp,
                'corrected_timestamp': corrected_time,
                'x_px': x,
                'y_px': y,
                'phase': phase
            })
            
        # Save results
        audio_name = processor.streams.get(processor.audio_id, {}).get('name', 'audio')
        output_file = output_dir / f"{audio_name}_corrected_timestamps.csv"
        
        with open(output_file, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['original_timestamp', 'corrected_timestamp', 
                                                   'x_px', 'y_px', 'phase'])
            writer.writeheader()
            writer.writerows(corrected_data)
            
        # Summary
        calib_events = sum(1 for d in corrected_data if d['phase'] == 'calibration')
        exp_events = sum(1 for d in corrected_data if d['phase'] == 'experiment')
        
        print(f"\n✅ Saved {len(corrected_data)} corrected timestamps")
        print(f"   During calibration: {calib_events}")
        print(f"   During experiment: {exp_events}")
        
        # Performance metrics
        elapsed = time.time() - start_time
        print(f"\n⚡ Performance:")
        print(f"   Processing time: {elapsed:.1f}s")
        print(f"   Speed: {file_size_mb/elapsed:.1f} MB/s")
        print(f"   Efficiency: Processed ~{processor.audio_seconds_collected/elapsed:.1f}s of audio per second")
        
        # Save summary
        summary_file = output_dir / f"{audio_name}_summary.txt"
        with open(summary_file, 'w') as f:
            f.write(f"Ultra-Efficient XDF Processing Summary\n")
            f.write(f"=====================================\n\n")
            f.write(f"File: {xdf_path.name} ({file_size_mb:.1f} MB)\n")
            f.write(f"Processing time: {elapsed:.1f}s\n")
            f.write(f"Audio processed: {processor.audio_seconds_collected:.1f}s (max: {MAX_AUDIO_SECONDS}s)\n")
            f.write(f"Calibration duration: {calib_duration:.3f}s\n")
            f.write(f"Mouse events: {len(corrected_data)}\n")
            f.write(f"  - During calibration: {calib_events}\n")
            f.write(f"  - During experiment: {exp_events}\n")
            f.write(f"\nNew timeline: t=0 at {processor.calibration_end_time:.6f}\n")

if __name__ == "__main__":
    # User configuration
    xdf_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf"
    output_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT"
    
    # Enable debugging if needed
    # DEBUG = True
    
    try:
        process_xdf_ultra_efficient(xdf_path, output_dir)
    except FileNotFoundError:
        print(f"❌ File not found: {xdf_path}")
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

ULTRA-EFFICIENT XDF CALIBRATION DETECTOR
Input: P001_20250506_171104_R001.xdf (964.6 MB)
Output: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT
🚀 Starting ultra-efficient XDF processing...
📊 Phase 1: Stream identification and minimal audio extraction
  Unknown chunk type 16188 (chunk #1)
  Unknown chunk type 63215 (chunk #2)
  Unknown chunk type 0 (chunk #3)
  Unknown chunk type 16628 (chunk #4)

✅ Processing complete!
   Total chunks read: 4
   Audio processed: 0.0s (target: ≤30s)
   Mouse events collected: 0
❌ Failed to process XDF file


In [7]:
#!/usr/bin/env python3
"""
Ultra-Efficient XDF Calibration Detector

Core principle: Extract ONLY first 30 seconds of audio to detect calibration,
then skip ALL remaining audio. Process mouse data separately.

Efficiency targets:
- Memory usage: <50MB peak
- Processing time: <60 seconds for 1GB files  
- Audio processed: ~0.1% of total (30s out of hours)
- Disk usage: 0 bytes (no temp files)
"""

import os
import struct
import csv
import gzip
from typing import Optional, Tuple, List, Dict, BinaryIO
import xml.etree.ElementTree as ET

try:
    import numpy as np
    HAS_NUMPY = True
except ImportError:
    HAS_NUMPY = False
    print("Warning: NumPy not available. Using fallback detection.")

# Configuration
MAX_AUDIO_SECONDS = 30  # Only process first 30 seconds
SAMPLE_RATE = 44100  # Expected sample rate
CALIBRATION_DURATION = 0.5  # Expected calibration duration
DEBUG = True  # Set True for verbose debugging

def debug_print(msg: str):
    """Print only if debugging enabled"""
    if DEBUG:
        print(msg)

class UltraEfficientXDFProcessor:
    """Minimal memory footprint XDF processor"""
    
    def __init__(self, filepath: str):
        self.filepath = filepath
        self.file: Optional[BinaryIO] = None
        self.streams: Dict[int, dict] = {}
        self.audio_id: Optional[int] = None
        self.mouse_id: Optional[int] = None
        self.audio_start_time: Optional[float] = None
        self.calibration_end_time: Optional[float] = None
        self.mouse_events: List[Tuple[float, float, float]] = []
        self.audio_buffer = []  # Temporary buffer for audio samples
        self.audio_seconds_collected = 0.0
        
    def __enter__(self):
        # Check if file is gzipped
        if self.filepath.endswith('.xdfz'):
            self.file = gzip.open(self.filepath, 'rb')
        else:
            self.file = open(self.filepath, 'rb')
            
        # Read and verify XDF header
        magic = self.file.read(4)
        if magic != b'XDF:':
            raise ValueError(f"Invalid XDF file signature: {magic}")
            
        # Read file header
        file_header = self._read_file_header()
        print(f"XDF file version: {file_header}")
        
        return self
        
    def __exit__(self, *args):
        if self.file:
            self.file.close()
            
    def _read_file_header(self) -> str:
        """Read XDF file header after magic bytes"""
        header_length = struct.unpack('<I', self.file.read(4))[0]
        header_data = self.file.read(header_length - 2)  # -2 for tag
        # Skip the tag (2 bytes) - should be 1 for file header
        tag = struct.unpack('<H', self.file.read(2))[0]
        if tag != 1:
            debug_print(f"Warning: Expected file header tag 1, got {tag}")
        
        # Parse XML header
        try:
            root = ET.fromstring(header_data.decode('utf-8'))
            version = root.findtext('.//version', 'Unknown')
            return version
        except:
            return "Unknown"
            
    def read_chunk(self) -> Optional[Tuple[int, int, bytes]]:
        """Read a complete chunk: (length, tag, data)"""
        # Read chunk length
        length_bytes = self.file.read(4)
        if len(length_bytes) < 4:
            return None
            
        try:
            length = struct.unpack('<I', length_bytes)[0]
        except struct.error:
            return None
            
        if length == 0:
            return (0, 0, b'')
            
        # Read tag
        tag_bytes = self.file.read(2)
        if len(tag_bytes) < 2:
            return None
            
        tag = struct.unpack('<H', tag_bytes)[0]
        
        # Read remaining data
        data_length = length - 2  # -2 for tag
        if data_length > 0:
            data = self.file.read(data_length)
            if len(data) < data_length:
                return None
        else:
            data = b''
            
        return (length, tag, data)
            
    def parse_header_minimal(self, data: bytes) -> dict:
        """Extract only essential info from stream header"""
        try:
            # First 4 bytes are stream ID
            stream_id = struct.unpack('<I', data[:4])[0]
            xml_data = data[4:]
            
            xml = ET.fromstring(xml_data.decode('utf-8'))
            info = {
                'id': stream_id,
                'type': xml.findtext('.//type', ''),
                'name': xml.findtext('.//name', ''),
                'channel_count': int(xml.findtext('.//channel_count', '1')),
                'nominal_srate': float(xml.findtext('.//nominal_srate', '44100'))
            }
            return info
        except Exception as e:
            debug_print(f"Error parsing header: {e}")
            return {'id': -1, 'type': '', 'name': '', 'channel_count': 1}
            
    def parse_samples(self, data: bytes, stream_info: dict) -> List[Tuple[float, List[float]]]:
        """Parse sample data based on stream format"""
        samples = []
        channels = stream_info['channel_count']
        
        # First 4 bytes are stream ID (already read)
        offset = 0
        
        # Each sample: 8 bytes timestamp + channels * 4 bytes data
        record_size = 8 + (channels * 4)
        
        while offset + record_size <= len(data):
            # Read timestamp
            timestamp = struct.unpack('<d', data[offset:offset+8])[0]
            offset += 8
            
            # Read channel values
            values = []
            for ch in range(channels):
                val = struct.unpack('<f', data[offset:offset+4])[0]
                values.append(val)
                offset += 4
                
            samples.append((timestamp, values))
            
        return samples
            
    def detect_calibration_numpy(self, samples: np.ndarray, timestamps: np.ndarray) -> Optional[float]:
        """Fast NumPy-based calibration detection"""
        if len(samples) < SAMPLE_RATE * 0.1:  # Need at least 100ms
            return None
            
        # Simple but effective: look for high energy bursts
        window = int(0.02 * SAMPLE_RATE)  # 20ms windows
        step = window // 4
        
        # Calculate windowed RMS energy
        high_energy_windows = []
        for i in range(0, len(samples) - window, step):
            chunk = samples[i:i+window, 0]  # Use first channel
            rms = np.sqrt(np.mean(chunk**2))
            if rms > 0.01:  # Energy threshold
                high_energy_windows.append(i)
                
        # If we found 3+ high energy windows in first second, likely calibration
        if len(high_energy_windows) >= 3:
            # Calibration end is ~500ms after first detection
            first_detection_idx = high_energy_windows[0]
            calib_end_idx = min(first_detection_idx + int(CALIBRATION_DURATION * SAMPLE_RATE), 
                               len(timestamps) - 1)
            return timestamps[calib_end_idx]
            
        return None
        
    def detect_calibration_simple(self, samples: List[List[float]], timestamps: List[float]) -> Optional[float]:
        """Simple fallback calibration detection"""
        if not samples or len(samples) < 1000:
            return None
            
        # Count high-amplitude samples
        high_count = sum(1 for s in samples[:int(SAMPLE_RATE * 0.5)] 
                        if abs(s[0]) > 0.01)
        
        # If >10% samples are high amplitude in first 500ms, assume calibration
        if high_count > len(samples[:int(SAMPLE_RATE * 0.5)]) * 0.1:
            # Return timestamp 500ms after start
            return timestamps[0] + CALIBRATION_DURATION
            
        return None
        
    def process_efficient(self) -> bool:
        """Ultra-efficient processing: minimal audio, all mouse data"""
        print("🚀 Starting ultra-efficient XDF processing...")
        
        chunks_processed = 0
        samples_chunks = 0
        headers_found = 0
        
        # Track chunk types seen
        chunk_types_seen = {}
        
        print("📊 Phase 1: Stream identification and minimal audio extraction")
        
        while True:
            chunk = self.read_chunk()
            if not chunk:
                break
                
            length, tag, data = chunk
            chunks_processed += 1
            
            # Track chunk types
            chunk_types_seen[tag] = chunk_types_seen.get(tag, 0) + 1
            
            if chunks_processed <= 10 or chunks_processed % 100 == 0:
                debug_print(f"Chunk #{chunks_processed}: tag={tag}, length={length}")
            
            if tag == 1:  # File header (already read)
                continue
                
            elif tag == 2:  # Stream header
                headers_found += 1
                info = self.parse_header_minimal(data)
                if info['id'] != -1:
                    self.streams[info['id']] = info
                    
                    if info['type'] == 'Audio':
                        self.audio_id = info['id']
                        print(f"✓ Audio stream found: {info['name']} (ID: {info['id']}, {info['channel_count']} channels, {info['nominal_srate']} Hz)")
                    elif info['type'] == 'MouseEvents':
                        self.mouse_id = info['id']
                        print(f"✓ Mouse stream found: {info['name']} (ID: {info['id']}, {info['channel_count']} channels)")
                    else:
                        print(f"  Other stream: {info['name']} (type: {info['type']}, ID: {info['id']})")
                        
            elif tag == 3:  # Samples
                samples_chunks += 1
                
                # First 4 bytes are stream ID
                if len(data) < 4:
                    continue
                    
                stream_id = struct.unpack('<I', data[:4])[0]
                sample_data = data[4:]
                
                if samples_chunks <= 5:
                    debug_print(f"  Sample chunk #{samples_chunks}: stream_id={stream_id}, size={len(sample_data)} bytes")
                
                # Process audio if it's our audio stream
                if stream_id == self.audio_id and self.audio_id is not None:
                    if self.audio_seconds_collected < MAX_AUDIO_SECONDS:
                        stream_info = self.streams[self.audio_id]
                        samples = self.parse_samples(sample_data, stream_info)
                        
                        if samples:
                            # Set start time from first sample
                            if self.audio_start_time is None:
                                self.audio_start_time = samples[0][0]
                                print(f"⏰ Audio start time: {self.audio_start_time:.6f}")
                            
                            # Collect samples up to limit
                            for timestamp, values in samples:
                                if timestamp - self.audio_start_time > MAX_AUDIO_SECONDS:
                                    break
                                self.audio_buffer.append((timestamp, values))
                            
                            if self.audio_buffer:
                                self.audio_seconds_collected = self.audio_buffer[-1][0] - self.audio_start_time
                                
                                if samples_chunks % 10 == 0 or self.audio_seconds_collected >= 1.0:
                                    print(f"  Audio progress: {self.audio_seconds_collected:.1f}s collected ({len(self.audio_buffer)} samples)")
                            
                            # Try calibration detection at 1s, 5s, 10s
                            if self.audio_seconds_collected >= 1.0 and not self.calibration_end_time:
                                print(f"🔍 Attempting calibration detection with {len(self.audio_buffer)} samples...")
                                
                                timestamps = [s[0] for s in self.audio_buffer]
                                
                                if HAS_NUMPY:
                                    samples_array = np.array([s[1] for s in self.audio_buffer], dtype=np.float32)
                                    self.calibration_end_time = self.detect_calibration_numpy(
                                        samples_array, np.array(timestamps))
                                else:
                                    samples_list = [s[1] for s in self.audio_buffer]
                                    self.calibration_end_time = self.detect_calibration_simple(
                                        samples_list, timestamps)
                                
                                if self.calibration_end_time:
                                    calib_duration = self.calibration_end_time - self.audio_start_time
                                    print(f"✅ Calibration detected! Duration: {calib_duration:.3f}s")
                                    print(f"⚡ Clearing audio buffer to save memory...")
                                    self.audio_buffer = []  # Free memory
                                    
                elif stream_id == self.mouse_id and self.mouse_id is not None:
                    # Always collect mouse data
                    stream_info = self.streams[self.mouse_id]
                    samples = self.parse_samples(sample_data, stream_info)
                    
                    for timestamp, values in samples:
                        # Mouse data: [rel_time, x, y]
                        x = values[1] if len(values) > 1 else 0
                        y = values[2] if len(values) > 2 else 0
                        self.mouse_events.append((timestamp, x, y))
                        
            elif tag == 4:  # Timestamp
                continue
                
            elif tag == 5:  # Chunk index
                continue
                
            elif tag == 6:  # Stream footer
                continue
                
            # Stop if we have what we need from phase 1
            if (self.calibration_end_time or self.audio_seconds_collected >= MAX_AUDIO_SECONDS) and self.audio_start_time:
                print(f"\n⚡ Phase 1 complete! Processed {chunks_processed} chunks")
                break
                
            # Safety limit
            if chunks_processed > 10000:
                print(f"\n⚠️ Safety limit reached after {chunks_processed} chunks")
                break
                
        # Phase 2: Continue for remaining mouse data (skip audio)
        if self.audio_start_time and chunks_processed < 100000:  # Reasonable limit
            print("\n📊 Phase 2: Collecting remaining mouse data (skipping audio)")
            
            phase2_chunks = 0
            phase2_mouse_events = len(self.mouse_events)
            
            while True:
                chunk = self.read_chunk()
                if not chunk:
                    break
                    
                length, tag, data = chunk
                phase2_chunks += 1
                
                if tag == 3:  # Samples
                    if len(data) >= 4:
                        stream_id = struct.unpack('<I', data[:4])[0]
                        
                        if stream_id == self.mouse_id and self.mouse_id is not None:
                            stream_info = self.streams[self.mouse_id]
                            samples = self.parse_samples(data[4:], stream_info)
                            
                            for timestamp, values in samples:
                                x = values[1] if len(values) > 1 else 0
                                y = values[2] if len(values) > 2 else 0
                                self.mouse_events.append((timestamp, x, y))
                                
                if phase2_chunks % 1000 == 0:
                    new_events = len(self.mouse_events) - phase2_mouse_events
                    debug_print(f"  Phase 2 progress: {phase2_chunks} chunks, {new_events} new mouse events")
                    
        # Use default calibration if needed
        if not self.calibration_end_time and self.audio_start_time:
            self.calibration_end_time = self.audio_start_time + CALIBRATION_DURATION
            print(f"\n⚠️ Using default calibration duration: {CALIBRATION_DURATION}s")
            
        print(f"\n✅ Processing complete!")
        print(f"   Total chunks: {chunks_processed + phase2_chunks if 'phase2_chunks' in locals() else chunks_processed}")
        print(f"   Chunk types seen: {chunk_types_seen}")
        print(f"   Stream headers found: {headers_found}")
        print(f"   Sample chunks: {samples_chunks}")
        print(f"   Audio processed: {self.audio_seconds_collected:.1f}s")
        print(f"   Mouse events: {len(self.mouse_events)}")
        
        return bool(self.calibration_end_time)

def process_xdf_ultra_efficient(xdf_path: str, output_dir: str):
    """Main processing function - ultra efficient approach"""
    import pathlib
    import time
    
    start_time = time.time()
    
    print("="*60)
    print("ULTRA-EFFICIENT XDF CALIBRATION DETECTOR")
    print("="*60)
    
    # Setup paths
    xdf_path = pathlib.Path(xdf_path)
    output_dir = pathlib.Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    file_size_mb = xdf_path.stat().st_size / 1024 / 1024
    print(f"Input: {xdf_path.name} ({file_size_mb:.1f} MB)")
    print(f"Output: {output_dir}")
    
    # Process file
    with UltraEfficientXDFProcessor(str(xdf_path)) as processor:
        success = processor.process_efficient()
        
        if not success:
            print("❌ Failed to process XDF file")
            return
            
        # Extract results
        calib_duration = processor.calibration_end_time - processor.audio_start_time
        
        print(f"\n📊 Results:")
        print(f"   Calibration end: {processor.calibration_end_time:.6f}")
        print(f"   Calibration duration: {calib_duration:.3f}s")
        print(f"   New t=0: {processor.calibration_end_time:.6f}")
        
        # Correct mouse timestamps
        corrected_data = []
        for timestamp, x, y in processor.mouse_events:
            corrected_time = timestamp - processor.calibration_end_time
            phase = 'calibration' if corrected_time < 0 else 'experiment'
            corrected_data.append({
                'original_timestamp': timestamp,
                'corrected_timestamp': corrected_time,
                'x_px': x,
                'y_px': y,
                'phase': phase
            })
            
        # Save results
        audio_name = processor.streams.get(processor.audio_id, {}).get('name', 'audio')
        output_file = output_dir / f"{audio_name}_corrected_timestamps.csv"
        
        with open(output_file, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['original_timestamp', 'corrected_timestamp', 
                                                   'x_px', 'y_px', 'phase'])
            writer.writeheader()
            writer.writerows(corrected_data)
            
        # Summary
        calib_events = sum(1 for d in corrected_data if d['phase'] == 'calibration')
        exp_events = sum(1 for d in corrected_data if d['phase'] == 'experiment')
        
        print(f"\n✅ Saved {len(corrected_data)} corrected timestamps")
        print(f"   During calibration: {calib_events}")
        print(f"   During experiment: {exp_events}")
        
        # Performance metrics
        elapsed = time.time() - start_time
        print(f"\n⚡ Performance:")
        print(f"   Processing time: {elapsed:.1f}s")
        print(f"   Speed: {file_size_mb/elapsed:.1f} MB/s")
        print(f"   Efficiency: Processed only {processor.audio_seconds_collected:.1f}s of audio")
        
        # Save summary
        summary_file = output_dir / f"{audio_name}_summary.txt"
        with open(summary_file, 'w') as f:
            f.write(f"Ultra-Efficient XDF Processing Summary\n")
            f.write(f"=====================================\n\n")
            f.write(f"File: {xdf_path.name} ({file_size_mb:.1f} MB)\n")
            f.write(f"Processing time: {elapsed:.1f}s\n")
            f.write(f"Audio processed: {processor.audio_seconds_collected:.1f}s (max: {MAX_AUDIO_SECONDS}s)\n")
            f.write(f"Calibration duration: {calib_duration:.3f}s\n")
            f.write(f"Mouse events: {len(corrected_data)}\n")
            f.write(f"  - During calibration: {calib_events}\n")
            f.write(f"  - During experiment: {exp_events}\n")
            f.write(f"\nNew timeline: t=0 at {processor.calibration_end_time:.6f}\n")

if __name__ == "__main__":
    # User configuration
    xdf_path = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level2_RunExperiment\Results\LSL_Output\P001_20250506_171104_R001.xdf"
    output_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT"
    
    try:
        process_xdf_ultra_efficient(xdf_path, output_dir)
    except FileNotFoundError:
        print(f"❌ File not found: {xdf_path}")
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

ULTRA-EFFICIENT XDF CALIBRATION DETECTOR
Input: P001_20250506_171104_R001.xdf (964.6 MB)
Output: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level3_Analysis\Output\PPS_RT
XDF file version: Unknown
🚀 Starting ultra-efficient XDF processing...
📊 Phase 1: Stream identification and minimal audio extraction
Chunk #1: tag=63215, length=101234688
Chunk #2: tag=0, length=61

✅ Processing complete!
   Total chunks: 2
   Chunk types seen: {63215: 1, 0: 1}
   Stream headers found: 0
   Sample chunks: 0
   Audio processed: 0.0s
   Mouse events: 0
❌ Failed to process XDF file
